# Risk-controlled screening of AI coding agent delegation on GitHub

This notebook runs the whole study from raw data to final figures. It downloads the AIDev dataset, builds the issue-level table, trains the success model month by month, picks the delegation threshold with conformal risk control, and then runs the analyses and figures used in the thesis.

**How to run:** Runtime → Change runtime type → **T4 GPU**, then Runtime → **Run all**. The full run takes about 90 minutes without step 6b, and about 2.5 hours with it. Only steps 6 and 6b need the GPU. Step 6b and the bi-weekly evaluation in step 10 are the slowest parts.

**The question.** Every issue in this data was handed to an AI coding agent (Claude Code, Copilot, Codex, …), which then opened a pull request. We ask: *before the agent starts, can we tell which issues are worth delegating, while keeping the number of failed agent PRs under a chosen budget?*

**Three numbers come up everywhere:**
- **Coverage**: the share of issues we decide to delegate.
- **Risk**: the share of *all* issues that were delegated *and* failed. This is what the budget α (0.10) controls.
- **Failure rate among delegated**: risk ÷ coverage, i.e. 1 − precision.

| Step | What it does |
|---|---|
| 0 | Install libraries and print versions |
| 1 | Build the dataset (14-day merge label, open PRs handled) |
| 2 | Rolling monthly deployment: features, model, thresholds, baselines |
| 3 | SHAP: which features the model relies on |
| 4 | Confidence intervals and within-repository evaluation |
| 5 | Robustness checks |
| 6 | Zero-shot LLM baseline (GPU) |
| 6b | The same test with four more open LLMs (GPU) |
| 7 | Sample of 200 failed PRs for manual label checking |
| 8 | Pooled vs within-repository AUC of each feature |
| 9 | Results across different risk budgets |
| 10 | Bi-weekly deployment, simpler policies, safety margin, online adjustment, cold start |
| 11 | Look at the PR timeline table |
| 12 | Accuracy, precision, recall, F1 |
| 13 | All thesis figures |
| 14 | Warm-up policy for repositories with no agent history |
| 15 | Checking the decision time: PR timeline and a sensitivity test |
| 16 | Separate thresholds per group (hierarchical risk control) |
| 17 | Online risk control using early review signals |
| 18 | Asymmetric online risk control |
| 19 | Repository-level information and project-level decisions |
| 20 | Ablation of repository information and the combined policy |
| 21 | Save everything to Google Drive |

Steps 3–20 reuse variables from step 2, and steps 14–20 also reuse the bi-weekly models from step 10 (steps 18 and 20 also need step 17, and step 20 needs step 19), so run them in the same session and in order.

## Step 0: Setup
Installs the libraries Colab doesn't have (LightGBM, SHAP, Hugging Face tools) and prints the version of each one. The versions go in the thesis appendix so the run can be reproduced. It also checks whether a GPU is attached.

In [ ]:
!pip -q install huggingface_hub pyarrow lightgbm transformers accelerate shap
import sys, sklearn, lightgbm, pandas, numpy, scipy, shap, transformers, torch
print("Python", sys.version.split()[0])
for mod in [pandas, numpy, sklearn, lightgbm, scipy, shap, transformers, torch]:
    print(f"{mod.__name__:14s} {mod.__version__}")
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (llm_res needs a T4 GPU)")

## Step 1: Build the dataset
Downloads three AIDev tables (pull requests, issues, and the links between them) and turns them into one row per issue.

What the cell does, in order:
1. **Finds the data snapshot date**, the latest timestamp in the PR table, and prints how many PRs from each month were still open at that date. PRs from the last few months are often still open, so their outcome is unknown.
2. **Defines the label.** An issue counts as a success (`label = 1`) if the agent's PR was **merged within 14 days** of being opened. To make sure every label is final, it keeps only PRs opened at least 14 days before the snapshot. Still-open PRs are kept and count as not merged within 14 days.
3. **Keeps only clean links.** It drops PRs linked to more than one issue, issues created after the PR, and very short issue texts. If several agents attempted the same issue, it keeps only the first attempt.
4. **Records when each outcome became known** (`known_at`). Step 2 uses this so that no feature ever uses an outcome from the future.
5. **Flags two groups:**
   - *self-delegated*: the issue author opened the PR themselves.
   - *leak-flagged*: the issue text already mentions the fix ("fixed in #…", "PR #…").
   
   Both are removed from the main set but kept in `routing_all_v2.parquet` for the robustness checks.

**Output:** `routing_main_v2.parquet` (main set, about 30.6k issues) and `routing_all_v2.parquet`.

In [ ]:
# Label = merged within H days of the PR being opened. PRs that are still open count as
# not merged, and we only keep PRs opened at least H days before the snapshot.
import numpy as np, pandas as pd, warnings; warnings.filterwarnings("ignore")
from huggingface_hub import hf_hub_download, HfApi
dl = lambda f: hf_hub_download("hao-li/AIDev", f, repo_type="dataset")
H = 14                                                                # follow-up horizon in days
print("AIDev commit:", HfApi().dataset_info("hao-li/AIDev").sha)

ri  = pd.read_parquet(dl("related_issue.parquet"))
iss = pd.read_parquet(dl("issue.parquet"), columns=["id", "title", "body", "user", "created_at"])
pr  = pd.read_parquet(dl("pull_request.parquet"), columns=["id", "agent", "user", "state", "created_at", "merged_at", "closed_at",
                                                          "repo_id", "repo_url", "html_url"]).drop_duplicates("id")
for c in ["created_at", "merged_at", "closed_at"]: pr[c] = pd.to_datetime(pr[c], utc=True, errors="coerce")
iss["created_at"] = pd.to_datetime(iss["created_at"], utc=True, errors="coerce")
SNAP = pd.concat([pr.created_at, pr.merged_at, pr.closed_at]).max()
print("Data snapshot (latest timestamp in the PR table):", SNAP)

# Censoring report: share of PRs still open, by month of creation
pr["month"] = pr.created_at.dt.strftime("%Y-%m")
print("\nShare of PRs still open at the snapshot, by month:\n",
      pr.groupby("month").apply(lambda g: pd.Series(dict(prs=len(g), open_share=(g.merged_at.isna() & g.closed_at.isna()).mean()))).tail(12).round(3).to_string())
lag = (pr.merged_at - pr.created_at).dt.total_seconds() / 86400
print(f"\nMerged PRs merged within {H} days: {(lag[pr.merged_at.notna()] <= H).mean():.3f}")

# Build the issue-level table: one issue, its first agent PR, and the outcome
nl = ri.groupby("pr_id").issue_id.transform("nunique"); ri1 = ri[nl == 1]
d = (ri1.merge(pr.rename(columns={"id": "pr_id", "created_at": "pr_created", "user": "pr_user"}), on="pr_id")
        .merge(iss.rename(columns={"id": "issue_id", "created_at": "issue_created", "user": "issue_user"}), on="issue_id"))
d = d[d.issue_created < d.pr_created]
d["text"] = (d.title.fillna("") + "\n\n" + d.body.fillna("")).str.strip(); d = d[d.text.str.len() >= 20]
d = d.sort_values("pr_created").drop_duplicates("issue_id", keep="first")
d = d[d.pr_created <= SNAP - pd.Timedelta(days=H)]                    # full follow-up only
hdl = d.pr_created + pd.Timedelta(days=H)
merged_in_h = d.merged_at.notna() & (d.merged_at <= hdl)
d["label"] = merged_in_h.astype(int)
d["label_eventual"] = d.merged_at.notna().astype(int)
d["known_at"] = np.where(merged_in_h, d.merged_at, np.where(d.closed_at.notna() & (d.closed_at < hdl), d.closed_at, hdl))
d["known_at"] = pd.to_datetime(d.known_at, utc=True)
d["self_delegated"] = d.issue_user.astype(str) == d.pr_user.astype(str)
d["leak_flag"] = d.text.str.contains(r"pull/\d+|fixed (?:in|by) #|merged (?:in|into)|\bPR #\d+", case=False, regex=True)
d["bot_issue"] = d.issue_user.astype(str).str.contains("[bot]", regex=False)
print(f"\nIssues with full follow-up: {len(d):,} | label differs from eventual merge: {(d.label != d.label_eventual).mean():.3f}")
print("Agent mix and merge-within-H by group:\n", d.groupby(["self_delegated"]).label.agg(["size", "mean"]).round(3).to_string())

cols = ["issue_id", "pr_id", "agent", "repo_id", "repo_url", "html_url", "issue_created", "pr_created", "known_at", "text",
        "bot_issue", "label", "label_eventual", "self_delegated", "leak_flag", "issue_user", "pr_user"]
d[cols].to_parquet("routing_all_v2.parquet", index=False)
main = d[~d.self_delegated & ~d.leak_flag][cols].reset_index(drop=True)
main.to_parquet("routing_main_v2.parquet", index=False)
print(f"\nMain set (not self-delegated, no leak flag): {len(main):,} issues | merge-within-{H}d rate {main.label.mean():.3f}")
print("By agent:\n", main.groupby("agent").label.agg(["size", "mean"]).round(3).to_string())

## Step 2: Rolling monthly deployment
The core experiment. For each month from **September 2025 to March 2026**, it simulates a team using the system at the start of the month:

| Period | Issues used | Purpose |
|---|---|---|
| Training | Every issue before the calibration window | Fit the model |
| Calibration | The 30 days ending 14 days before the month | Choose the threshold |
| Test | The month itself | Measure what actually happens |

The 14-day gap guarantees that every calibration outcome was known on the deployment day.

**Features**, all computed only from information known at the moment the agent was started:
- *Repository history*: the repository's past success rate, smoothed towards the overall rate. It is computed over all history and over the last 90 days, and the cell also records how many past outcomes the repository has.
- *Issue structure* (9 features): text length, title length, code blocks, stack trace, links, checklist items, reproduction steps, file paths, and whether a bot opened the issue.
- *Text score*: the output of a TF-IDF + logistic-regression model trained on issue text. On the training data it is computed out of fold, so the model never scores text it was trained on.

**Model:** LightGBM, retrained every month.

**Threshold:** chosen on the calibration window with two methods:
- **CRC** (conformal risk control) aims for average risk ≤ α.
- **LTT** (Learn-then-Test) aims for risk ≤ α with 90% confidence.

**Compared against:** a model trained once and never updated, repository history only, recent agent success rate, issue structure only, and random scores.

The cell also tests:
- separate thresholds per repository and per agent;
- controlling the failure rate *among delegated* issues instead;
- how small the calibration set can be (the last 50/100/250/500 issues).

**Output:** `results_rolling_*.csv`. `results_rolling_predictions.csv` holds every test issue with its score and decision and is used by the later steps.

In [ ]:
import numpy as np, pandas as pd, lightgbm as lgb, warnings; warnings.filterwarnings("ignore")
from huggingface_hub import hf_hub_download
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from scipy.stats import binom

dl = lambda f: hf_hub_download("hao-li/AIDev", f, repo_type="dataset")
main = pd.read_parquet("routing_main_v2.parquet")                    # built in step 1
main["resolved_at"] = pd.to_datetime(main["known_at"], utc=True)      # when the label became known

# Decision time = when the agent was started (PR creation). Outcomes count as known only after resolved_at.
ts = lambda s: s.dt.tz_convert("UTC").dt.tz_localize(None).values.astype("datetime64[ns]").astype("int64")
DEC, RES, Y = ts(main.pr_created), ts(main.resolved_at), main.label.values.astype(float)
DAY = 86_400 * 10**9

def known_before(groups, window_days=None):
    """Merges and count among earlier issues in the same group whose outcome was known before each decision."""
    s, c = np.zeros(len(main)), np.zeros(len(main))
    for _, idx in pd.Series(np.arange(len(main))).groupby(np.asarray(groups)).indices.items():
        o = np.argsort(RES[idx]); rs = RES[idx][o]; cy = np.concatenate([[0.0], np.cumsum(Y[idx][o])])
        hi = np.searchsorted(rs, DEC[idx], side="left")
        lo = np.zeros_like(hi) if window_days is None else np.searchsorted(rs, DEC[idx] - window_days * DAY, side="left")
        s[idx], c[idx] = cy[hi] - cy[lo], hi - lo
    return s, c

sg, cg = known_before(np.zeros(len(main)))
G = np.where(cg > 0, sg / np.maximum(cg, 1), 0.5)                    # global merge rate known so far
sr, cr = known_before(main.repo_id.values);              REPO_PRIOR = (sr + 10 * G) / (cr + 10); REPO_N = np.log1p(cr)
sr9, cr9 = known_before(main.repo_id.values, 90);        REPO_PRIOR90 = (sr9 + 10 * G) / (cr9 + 10)
sa, ca_ = known_before(main.agent.values, 30);           AGENT_RATE30 = (sa + 10 * G) / (ca_ + 10)
REPO_HIST = cr                                                       # number of known past outcomes in the repo

t = main.text
ISS = np.column_stack([
    np.log1p(t.str.len()), t.str.split("\n").str[0].str.len(), t.str.count("```") // 2,
    t.str.contains(r"Traceback|Exception|Error:|at \S+\(\S+:\d+\)", regex=True),
    t.str.count(r"https?://"), t.str.count(r"- \[[ xX]\]"),
    t.str.contains(r"steps to reproduce|to reproduce|expected behavio", case=False, regex=True),
    t.str.count(r"\b[\w/]+\.(?:py|js|ts|tsx|java|go|rs|cpp|c|rb|md|ya?ml|json)\b"),
    main.bot_issue]).astype(float)

def tfidf_scores(tr, other, folds=3, seed=0):
    out = np.zeros(len(main)); txt = main.text.values
    def fit(a, b):
        v = TfidfVectorizer(max_features=50000, ngram_range=(1, 2), min_df=3, sublinear_tf=True)
        lr = LogisticRegression(C=1.0, max_iter=2000).fit(v.fit_transform(txt[a]), Y[a])
        out[b] = lr.predict_proba(v.transform(txt[b]))[:, 1]
    for a, b in StratifiedKFold(folds, shuffle=True, random_state=seed).split(tr, Y[tr]): fit(tr[a], tr[b])
    fit(tr, other); return out

def lgbm(X, tr):
    return lgb.LGBMClassifier(n_estimators=400, learning_rate=0.03, num_leaves=15, min_child_samples=50,
                              subsample=0.8, subsample_freq=1, random_state=42, verbose=-1).fit(X[tr], Y[tr])

GRID, CGRID = np.linspace(0, 1, 1001), np.linspace(0, 1, 101)
def fails_above(p, y):
    pf = np.sort(p[y == 0]); return len(pf) - np.searchsorted(pf, GRID, side="left")
def lam_crc(p, y, a):                                   # E[delegated AND failed] <= a
    n = len(p); ok = np.where(fails_above(p, y) / (n + 1) + 1 / (n + 1) <= a)[0]; return GRID[ok[0]] if len(ok) else 1.01
def lam_ltt(p, y, a, d=0.1):                            # P(risk <= a) >= 1 - d
    k, lam = fails_above(p, y), 1.01
    for i in range(len(GRID) - 1, -1, -1):
        if binom.cdf(k[i], len(p), a) <= d: lam = GRID[i]
        else: break
    return lam
def lam_cond(p, y, b, d=0.1):                           # P(fail | delegated) <= b, Bonferroni over a coarse grid
    for lam in CGRID:
        sel = p >= lam
        if sel.sum() and binom.cdf((y[sel] == 0).sum(), sel.sum(), b) <= d / len(CGRID): return lam
    return 1.01
def stats(p, y, lam):
    sel = p >= lam
    return dict(coverage=sel.mean(), risk=np.mean(sel & (y == 0)), cond_fail=(1 - y[sel].mean()) if sel.any() else np.nan)

# ---------------- rolling monthly deployment ----------------
starts = pd.date_range("2025-09-01", "2026-03-01", freq="MS", tz="UTC")
ALPHA, rng = 0.10, np.random.default_rng(0)
rows, sizes, agents_rows, cond_rows, records, repo_rows = [], [], [], [], [], []
feat_main = lambda TXT: np.column_stack([REPO_PRIOR, REPO_N, REPO_PRIOR90, ISS, TXT])
M0 = None; models_store = []                                      # kept for the SHAP cell
for w, t0 in enumerate(starts):
    t0i = t0.value; t1i = (t0 + pd.offsets.MonthBegin(1)).value
    c1 = t0i - 14 * DAY; c0 = c1 - 30 * DAY                          # calibration: 30 days ending H = 14 days before deployment
    tr = np.where(DEC < c0)[0]                                       # every label known by c0 + H < t0
    ca = np.where((DEC >= c0) & (DEC < c1))[0]                       # every label known by t0, so no selection on fast outcomes
    te = np.where((DEC >= t0i) & (DEC < t1i))[0]
    if len(te) < 50 or len(ca) < 50: continue
    other = np.concatenate([ca, te])
    TXT = tfidf_scores(tr, other); X = feat_main(TXT); m = lgbm(X, tr)
    Xa = np.column_stack([X, AGENT_RATE30]); ma = lgbm(Xa, tr)
    ms = lgbm(ISS, tr)
    models_store.append((t0.strftime("%Y-%m"), m, X, te))
    if M0 is None:                                                   # fixed model trained once, at the first window
        TXT0 = tfidf_scores(tr, np.arange(len(main))); X0 = feat_main(TXT0); M0 = lgbm(X0, tr)
        p0_ca = M0.predict_proba(X0[ca])[:, 1]; lam0 = lam_crc(p0_ca, Y[ca], ALPHA)
    scores = {
        "Main: retrained + recalibrated": m.predict_proba(X)[:, 1],
        "Main + recent agent rate": ma.predict_proba(Xa)[:, 1],
        "Fixed model + recalibrated": M0.predict_proba(X0)[:, 1],
        "Baseline: repo history only": REPO_PRIOR,
        "Baseline: repo history, ties broken": REPO_PRIOR + 1e-6 * rng.random(len(main)),
        "Baseline: recent agent rate": AGENT_RATE30,
        "Baseline: issue structure only": ms.predict_proba(ISS)[:, 1],
        "Baseline: random": rng.random(len(main)),
    }
    for name, p in scores.items():
        for meth, f in [("CRC", lam_crc), ("LTT", lam_ltt)]:
            r = dict(window=t0.strftime("%Y-%m"), policy=name, method=meth, n_train=len(tr), n_calib=len(ca), n_test=len(te),
                     test_merge=Y[te].mean(), random_cov_ref=min(1, ALPHA / (1 - Y[ca].mean())), auc=roc_auc_score(Y[te], p[te]) if name != "Baseline: random" else 0.5,
                     **stats(p[te], Y[te], f(p[ca], Y[ca], ALPHA)))
            rows.append(r)
    rows.append(dict(window=t0.strftime("%Y-%m"), policy="Fixed model + fixed threshold", method="CRC", n_train=len(tr),
                     n_calib=len(ca), n_test=len(te), test_merge=Y[te].mean(),
                     auc=roc_auc_score(Y[te], scores["Fixed model + recalibrated"][te]), **stats(scores["Fixed model + recalibrated"][te], Y[te], lam0)))
    pm = scores["Main: retrained + recalibrated"]
    lam_main = lam_crc(pm[ca], Y[ca], ALPHA)
    records.append(pd.DataFrame(dict(window=t0.strftime("%Y-%m"), y=Y[te], p_main=pm[te], p_repo=REPO_PRIOR[te],
                                     delegated=pm[te] >= lam_main, repo=main.repo_id.values[te], agent=main.agent.values[te],
                                     issue_id=main.issue_id.values[te], repo_hist=REPO_HIST[te])))
    # conditional failure control
    for b_ in [0.20, 0.25]:
        cond_rows.append(dict(window=t0.strftime("%Y-%m"), beta=b_, **stats(pm[te], Y[te], lam_cond(pm[ca], Y[ca], b_))))
    # chronological calibration size: the n most recent issues with known outcomes
    known = ca[np.argsort(DEC[ca])]                                  # never overlaps the training set
    for n in [50, 100, 250, 500]:
        if n > len(known): continue
        cn = known[-n:]
        for meth, f in [("CRC", lam_crc), ("LTT", lam_ltt)]:
            sizes.append(dict(window=t0.strftime("%Y-%m"), n_calib=n, method=meth, **stats(pm[te], Y[te], f(pm[cn], Y[cn], ALPHA))))
    # per-repository (group-conditional) thresholds: own threshold if >= 30 calibration issues, else global
    lam_repo = {}
    for rp, pos in pd.Series(ca).groupby(main.repo_id.values[ca]).indices.items():
        cix = ca[pos]
        if len(cix) >= 30: lam_repo[rp] = lam_crc(pm[cix], Y[cix], ALPHA)
    lam_vec = np.array([lam_repo.get(rp, lam_main) for rp in main.repo_id.values[te]])
    for rp in np.unique(main.repo_id.values[te]):
        tix = te[main.repo_id.values[te] == rp]
        if len(tix) < 20: continue
        for kind, lam in [("global threshold", lam_main), ("repo-specific threshold", lam_repo.get(rp, lam_main))]:
            repo_rows.append(dict(window=t0.strftime("%Y-%m"), repo=rp, threshold=kind, own_threshold=rp in lam_repo,
                                  n_test=len(tix), **stats(pm[tix], Y[tix], lam)))
    rows.append(dict(window=t0.strftime("%Y-%m"), policy="Main + repo-specific thresholds", method="CRC", n_train=len(tr),
                     n_calib=len(ca), n_test=len(te), test_merge=Y[te].mean(), random_cov_ref=min(1, ALPHA / (1 - Y[ca].mean())),
                     auc=roc_auc_score(Y[te], pm[te]), coverage=np.mean(pm[te] >= lam_vec), risk=np.mean((pm[te] >= lam_vec) & (Y[te] == 0)),
                     cond_fail=(1 - Y[te][pm[te] >= lam_vec].mean()) if (pm[te] >= lam_vec).any() else np.nan))
    # per-agent: global threshold vs agent-specific thresholds (risk measured within each agent)
    for ag in ["Claude_Code", "Copilot", "OpenAI_Codex"]:
        ca_a, te_a = ca[main.agent.values[ca] == ag], te[main.agent.values[te] == ag]
        if len(te_a) < 20: continue
        lam_a = lam_crc(pm[ca_a], Y[ca_a], ALPHA) if len(ca_a) >= 30 else 1.01
        for kind, lam in [("global threshold", lam_main), ("agent-specific threshold", lam_a)]:
            agents_rows.append(dict(window=t0.strftime("%Y-%m"), agent=ag, threshold=kind, n_test=len(te_a), n_calib=len(ca_a),
                                    **stats(pm[te_a], Y[te_a], lam)))

def frame(rows_, limit):
    df_ = pd.DataFrame(rows_)
    if len(df_): df_["violated"] = (df_.cond_fail if limit == "beta" else df_.risk) > (df_.beta if limit == "beta" else ALPHA)
    return df_
R, S, AG, CD = frame(rows, "a"), frame(sizes, "a"), frame(agents_rows, "a"), frame(cond_rows, "beta")
REC = pd.concat(records, ignore_index=True)
RP = frame(repo_rows, "a")
print("Per-window results (alpha = 0.10):\n", R.round(3).to_string(index=False))
summ = R.groupby(["policy", "method"]).agg(windows=("risk", "size"), mean_auc=("auc", "mean"), mean_coverage=("coverage", "mean"),
        mean_risk=("risk", "mean"), max_risk=("risk", "max"), mean_cond_fail=("cond_fail", "mean"), windows_violated=("violated", "sum"))
print("\nSummary over windows:\n", summ.round(3).to_string())
print("\nChronological calibration size (mean over windows):\n",
      S.groupby(["n_calib", "method"]).agg(mean_coverage=("coverage", "mean"), mean_risk=("risk", "mean"), windows_violated=("violated", "sum")).round(3).to_string())
if len(AG): print("\nPer agent (risk within agent):\n", AG.groupby(["agent", "threshold"]).agg(mean_coverage=("coverage", "mean"), mean_risk=("risk", "mean"),
      mean_cond_fail=("cond_fail", "mean"), windows_violated=("violated", "sum")).round(3).to_string())
print("\nConditional failure control, P(fail | delegated) <= beta:\n", CD.round(3).to_string(index=False))
if len(RP): print("\nPer-repository realised risk (repos with >= 20 test issues in a window):\n",
      RP.groupby("threshold").agg(repo_windows=("risk", "size"), mean_risk=("risk", "mean"), share_violating=("violated", "mean"),
      p90_risk=("risk", lambda x: x.quantile(0.9)), mean_coverage=("coverage", "mean")).round(3).to_string())
for df_, f in [(RP, "rolling_per_repo"), (R, "rolling_windows"), (S, "rolling_calib_size"), (AG, "rolling_per_agent"), (CD, "rolling_conditional"), (REC, "rolling_predictions")]:
    df_.to_csv(f"results_{f}.csv", index=False)

## Step 3: SHAP explanations
SHAP values show how much each feature moves the model's prediction. The cell computes the average absolute SHAP value of every feature in every monthly model. It then reports what share of the total comes from the three repository features, as opposed to the issue features. A summary plot for the last month is saved as `fig7_shap.png`.

In [ ]:
# Uses the monthly models stored by step 2, so run that first.
import shap, matplotlib.pyplot as plt
FEATS = ["repo prior", "repo history size", "repo prior (90 days)", "text length", "title length", "code blocks",
         "stack trace", "links", "checklist items", "reproduction steps", "file paths", "bot author", "text score (TF-IDF)"]
REPO_F = ["repo prior", "repo history size", "repo prior (90 days)"]
per_window = {}
for win, mdl, Xw, tew in models_store:
    sv = shap.TreeExplainer(mdl).shap_values(Xw[tew])
    sv = sv[1] if isinstance(sv, list) else sv                       # older SHAP versions return one array per class
    per_window[win] = pd.Series(np.abs(sv).mean(0), index=FEATS)
SH = pd.DataFrame(per_window)
SH["mean"] = SH.mean(axis=1); SH = SH.sort_values("mean", ascending=False)
share = (SH.loc[REPO_F].sum() / SH.sum()).rename("repository share")
print("Mean |SHAP| per feature and window:\n", SH.round(4).to_string())
print("\nShare of attribution from repository features, by window:\n", share.round(3).to_string())
win, mdl, Xw, tew = models_store[-1]                                 # summary plot for the last window
sv = shap.TreeExplainer(mdl).shap_values(Xw[tew]); sv = sv[1] if isinstance(sv, list) else sv
shap.summary_plot(sv, Xw[tew], feature_names=FEATS, show=False)
plt.title(f"SHAP values, main model, {win}"); plt.savefig("fig7_shap.png", dpi=300, bbox_inches="tight"); plt.show()
SH.to_csv("results_shap.csv")

## Step 4: Confidence intervals and within-repository evaluation
Pools the test months and asks how certain the results are and where the predictive power comes from.

- **Repository-clustered bootstrap (1,000 resamples):** resamples whole repositories, not single issues, because issues from the same repository are not independent. It gives 95% intervals for AUC, coverage, risk and failure rate among delegated, and for the AUC difference between the main model and repository history.
- **Within-repository AUC:** computes AUC *inside* each repository with at least 20 test issues and both outcomes, then averages. This tells us whether the model can rank issues within one project, not just separate good projects from bad ones.
- **Ties:** counts how many issues share the same repository-history score. Large ties make that score hard to threshold finely.
- **By repository history:** results for repositories with 0, 1–10, 11–50 and more than 50 known past outcomes.
- **Risk–coverage curve:** the failure rate among delegated issues as more and more issues are delegated (`fig6_risk_coverage.png`).

In [ ]:
import matplotlib.pyplot as plt
REC = pd.read_csv("results_rolling_predictions.csv")
repos = REC.repo.unique(); by_repo = {r: g.index.values for r, g in REC.groupby("repo")}
yb, pmb, prb, dlb = REC.y.values, REC.p_main.values, REC.p_repo.values, REC.delegated.values
def metrics(ix):
    yy, d = yb[ix], dlb[ix]
    return dict(auc_main=roc_auc_score(yy, pmb[ix]), auc_repo=roc_auc_score(yy, prb[ix]), coverage=d.mean(),
                risk=np.mean(d & (yy == 0)), cond_fail=1 - yy[d].mean() if d.any() else np.nan, precision=yy[d].mean() if d.any() else np.nan)
# Within-repository AUC: computed per repository (>= 20 test issues, both outcomes), then averaged weighted by size
elig_rows = [dict(repo=r, n=len(g), auc_main=roc_auc_score(g.y, g.p_main), auc_repo=roc_auc_score(g.y, g.p_repo))
             for r, g in REC.groupby("repo") if len(g) >= 20 and g.y.nunique() == 2]
WR = pd.DataFrame(elig_rows)
point = metrics(np.arange(len(REC))); point["auc_diff"] = point["auc_main"] - point["auc_repo"]
if len(WR):
    point["within_repo_auc_main"] = np.average(WR.auc_main, weights=WR.n); point["within_repo_auc_repo"] = np.average(WR.auc_repo, weights=WR.n)
rngb = np.random.default_rng(1); boots = []
for _ in range(1000):
    ix = np.concatenate([by_repo[r] for r in rngb.choice(repos, size=len(repos), replace=True)])
    if len(np.unique(yb[ix])) < 2: continue
    b = metrics(ix); b["auc_diff"] = b["auc_main"] - b["auc_repo"]
    if len(WR):
        w_ = WR.sample(len(WR), replace=True, random_state=int(rngb.integers(1e9)))
        b["within_repo_auc_main"] = np.average(w_.auc_main, weights=w_.n); b["within_repo_auc_repo"] = np.average(w_.auc_repo, weights=w_.n)
    boots.append(b)
B = pd.DataFrame(boots)
ci = pd.DataFrame({"estimate": pd.Series(point), "ci_low": B.quantile(0.025), "ci_high": B.quantile(0.975)}).round(4)
print("Pooled rolling test windows (alpha = 0.10), 95% repository-clustered bootstrap CIs:\n", ci.to_string())
print(f"\nWithin-repository AUC uses {len(WR)} repositories ({int(WR.n.sum()) if len(WR) else 0:,} test issues) with >= 20 test issues and both outcomes;"
      f" share of these repositories with within-repo AUC of the main model > 0.5: {(WR.auc_main > 0.5).mean() if len(WR) else float('nan'):.2f}")
WR.to_csv("results_within_repo_auc.csv", index=False)

# Ties in the repository-history score (explains the step in the repo-only router's coverage)
vc = pd.Series(np.round(prb, 6)).value_counts()
print(f"Repository-history score: {len(vc):,} distinct values among {len(prb):,} test issues; "
      f"largest tie block = {vc.iloc[0]:,} issues ({vc.iloc[0] / len(prb):.1%}).")

bins = pd.cut(REC.repo_hist, [-1, 0, 10, 50, np.inf], labels=["new (0)", "1-10", "11-50", ">50"])
byhist = REC.groupby(bins).apply(lambda g: pd.Series(dict(n=len(g), merge_rate=g.y.mean(),
         auc_main=roc_auc_score(g.y, g.p_main) if g.y.nunique() > 1 else np.nan,
         auc_repo=roc_auc_score(g.y, g.p_repo) if g.y.nunique() > 1 else np.nan,
         coverage=g.delegated.mean(), risk=np.mean(g.delegated & (g.y == 0)))))
print("\nBy repository history known at decision time:\n", byhist.round(3).to_string())

# Selective-classification view: failure rate among delegated vs coverage (pooled rolling test set)
fig, ax = plt.subplots(figsize=(5.2, 3.6)); cov = np.linspace(0.05, 1, 96)
for col, lab in [("p_main", "Main model"), ("p_repo", "Repository history")]:
    s_ = REC[col].values + 1e-9 * rngb.random(len(REC)); o = np.argsort(-s_); fr = 1 - np.cumsum(yb[o]) / np.arange(1, len(yb) + 1)
    ax.plot(cov, [fr[max(int(c * len(yb)) - 1, 0)] for c in cov], label=lab)
ax.axhline(1 - yb.mean(), color="k", ls="--", lw=1, label="Random")
ax.set_xlabel("Coverage (share delegated)"); ax.set_ylabel("Failure rate among delegated"); ax.grid(alpha=.3); ax.legend(fontsize=8)
plt.savefig("fig6_risk_coverage.png", dpi=300, bbox_inches="tight"); plt.show()
ci.to_csv("results_bootstrap_ci.csv"); byhist.to_csv("results_by_repo_history.csv")

## Step 5: Robustness checks
Checks that the main findings are not artefacts of how the data were selected.

- **Agent drift within the same repositories:** compares each agent's success rate before 2026 and in 2026, using only repositories that used that agent in both periods. This separates real changes in the agent from changes in which projects use it.
- **Leakage filter:** how many issues it removes and how their success rate differs.
- **Agent mix:** how each filtering step changes the share of each agent.
- **Multi-issue PRs:** PRs linked to several issues (dropped) vs one issue (kept), compared on merge rate and agent mix.
- **14-day label vs eventual merge:** checks whether the results change if "eventually merged" is used instead.
- **Copilot excluded:** Copilot PRs are opened by a bot account, so self-delegation cannot be detected for them. This reruns the pooled numbers without Copilot.

In [ ]:
allv = pd.read_parquet("routing_all_v2.parquet")
mv = pd.read_parquet("routing_main_v2.parquet")
mv["period"] = pd.cut(pd.to_datetime(mv.pr_created, utc=True).dt.tz_localize(None),
                      [pd.Timestamp("2000-01-01"), pd.Timestamp("2026-01-01"), pd.Timestamp("2100-01-01")], labels=["before 2026", "2026"])
# Agent drift within repositories that used the same agent in both periods
rows3 = []
for ag, g in mv.groupby("agent"):
    both = g.groupby("repo_id").period.nunique(); both = both[both == 2].index
    gb = g[g.repo_id.isin(both)]
    if len(both) < 5: continue
    raw = g.groupby("period").label.mean(); within = gb.groupby("period").label.mean()
    repo_avg = gb.groupby(["period", "repo_id"]).label.mean().groupby("period").mean()
    rows3.append(dict(agent=ag, repos_in_both=len(both), raw_before=raw.get("before 2026"), raw_2026=raw.get("2026"),
                     same_repos_before=within.get("before 2026"), same_repos_2026=within.get("2026"),
                     repo_weighted_before=repo_avg.get("before 2026"), repo_weighted_2026=repo_avg.get("2026")))
print("Agent merge rates: all repositories vs repositories that used the agent in both periods:\n", pd.DataFrame(rows3).round(3).to_string(index=False))

# Leakage filter: what it removes, and outcome of flagged issues
nsd = allv[~allv.self_delegated]
print(f"\nLeak-flagged among non-self-delegated issues: {nsd.leak_flag.mean():.3f}; merge rate flagged {nsd[nsd.leak_flag].label.mean():.3f} vs others {nsd[~nsd.leak_flag].label.mean():.3f}")
# How each selection step changes the agent mix
mix = pd.concat({"all issues with follow-up": allv.agent.value_counts(normalize=True),
                 "not self-delegated": nsd.agent.value_counts(normalize=True),
                 "main set": mv.agent.value_counts(normalize=True)}, axis=1).round(3)
print("\nAgent mix through the selection steps:\n", mix.to_string())

# PRs linked to several issues (dropped) vs one issue (kept)
ri = pd.read_parquet(dl("related_issue.parquet"))
prs = pd.read_parquet(dl("pull_request.parquet"), columns=["id", "agent", "merged_at", "closed_at", "repo_id"]).drop_duplicates("id")
prs = prs[prs.merged_at.notna() | prs.closed_at.notna()].copy(); prs["merged"] = prs.merged_at.notna().astype(int)
prs["linked_issues"] = prs.id.map(ri.groupby("pr_id").issue_id.nunique()).fillna(0).astype(int)
lk = prs[prs.linked_issues > 0].copy(); lk["group"] = np.where(lk.linked_issues > 1, "several issues (dropped)", "one issue (kept)")
print("\nClosed PRs linked to one vs several issues:\n", lk.groupby("group").agg(prs=("id", "size"), merge_rate=("merged", "mean"),
      repos=("repo_id", "nunique")).round(3).to_string())
print(pd.crosstab(lk.agent, lk.group, normalize="columns").round(3).to_string())

# First attempt vs eventual merge of that attempt (censoring-aware label vs eventual label)
REC = pd.read_csv("results_rolling_predictions.csv").merge(mv[["issue_id", "label_eventual"]], on="issue_id", how="left")
dd = REC.delegated.astype(bool)
print(f"\nRolling test AUC, merged within H days: {roc_auc_score(REC.y, REC.p_main):.4f} | eventually merged: {roc_auc_score(REC.label_eventual, REC.p_main):.4f}")
print(f"Failure rate among delegated, within H days: {1 - REC.y[dd].mean():.4f} | eventual: {1 - REC.label_eventual[dd].mean():.4f}")

# Copilot sensitivity (undetectable self-delegation): pooled rolling results with and without Copilot test issues
for lab_, sub in [("all agents", REC), ("Copilot excluded", REC[REC.agent != "Copilot"])]:
    ds = sub.delegated.astype(bool)
    print(f"{lab_:17s} n={len(sub):6,d} merge={sub.y.mean():.3f} AUC={roc_auc_score(sub.y, sub.p_main):.4f} "
          f"coverage={ds.mean():.3f} risk={np.mean(ds & (sub.y == 0)):.4f} fail_among_delegated={1 - sub.y[ds].mean():.4f}")

## Step 6: Zero-shot LLM baseline (needs GPU)
Asks a small instruction-tuned LLM (**Qwen2.5-1.5B-Instruct**) whether an agent's PR for each issue would be merged. The answer probability ("Yes" vs "No") is used as a score. No fine-tuning is done.

The cell scores up to 800 calibration and 1,500 test issues from the last monthly window. It then sets thresholds with CRC at α = 0.05, 0.10 and 0.20, and compares the LLM with the main model and with repository history on the same issues.

Takes about 10–15 minutes on a T4. A 3B model also fits if you change `MID`, but it is roughly twice as slow.

In [ ]:
# Needs a GPU. Uses the last monthly window from step 2 (ca, te, pm).
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
MID = "Qwen/Qwen2.5-1.5B-Instruct"        # "Qwen/Qwen2.5-3B-Instruct" also fits a T4 and is likely stronger, ~2x slower
tok = AutoTokenizer.from_pretrained(MID); tok.padding_side = "left"
llm = AutoModelForCausalLM.from_pretrained(MID, torch_dtype=torch.float16, device_map="cuda").eval()
first_tok = lambda ws: sorted({tok.encode(w, add_special_tokens=False)[0] for w in ws})
YES, NO = first_tok(["Yes", " Yes", "yes"]), first_tok(["No", " No", "no"])

def prompt(text):
    msg = [{"role": "system", "content": "You are an experienced open-source maintainer triaging GitHub issues."},
           {"role": "user", "content": "Here is a GitHub issue.\n\n" + text[:2500] + "\n\nIf an AI coding agent such as Claude Code, "
            "GitHub Copilot or OpenAI Codex were asked to resolve this issue, is it likely that its pull request would be merged "
            "by the maintainers? Answer with Yes or No."}]
    return tok.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def llm_scores(texts, bs=8):
    out = []
    for i in range(0, len(texts), bs):
        enc = tok([prompt(t) for t in texts[i:i + bs]], return_tensors="pt", padding=True).to("cuda")
        lp = torch.log_softmax(llm(**enc).logits[:, -1, :].float(), -1)
        out.extend(torch.sigmoid(torch.logsumexp(lp[:, YES], -1) - torch.logsumexp(lp[:, NO], -1)).cpu().numpy().tolist())
        if i % 400 == 0: print(f"  scored {i + len(texts[i:i + bs])}/{len(texts)}")
    return np.array(out)

rng4 = np.random.default_rng(0)
cs = rng4.choice(ca, min(800, len(ca)), replace=False)                 # last window's calibration issues
tsx = rng4.choice(te, min(1500, len(te)), replace=False)               # last window's test issues
L_c, L_t = llm_scores(main.text.values[cs].tolist()), llm_scores(main.text.values[tsx].tolist())
tie = lambda v: v + 1e-6 * rng4.random(len(v))
rows4 = []
for name, pc, pt in [(f"LLM zero-shot ({MID.split('/')[-1]})", L_c, L_t), ("Main model", pm[cs], pm[tsx]),
                     ("Repository history (ties broken)", tie(REPO_PRIOR[cs]), tie(REPO_PRIOR[tsx]))]:
    for a_ in [0.05, 0.10, 0.20]:
        rows4.append(dict(model=name, alpha=a_, n_calib=len(cs), n_test=len(tsx), auc=roc_auc_score(Y[tsx], pt),
                          **stats(pt, Y[tsx], lam_crc(pc, Y[cs], a_))))
llm_res = pd.DataFrame(rows4); print(llm_res.round(3).to_string(index=False)); llm_res.to_csv("results_llm_baseline.csv", index=False)

## Step 6b: More open LLMs
Step 6 used one small model, so a weak result there could just mean the model was too small. This step asks **several open models** the same question about the **same issues**: the same 800 calibration and 1,500 test issues from March 2026 (seed 0), the same prompt, and the same Yes/No scoring. The results are directly comparable to each other and to step 6.

| Model | Maker / licence | Why it's included | Loaded as |
|---|---|---|---|
| Qwen2.5-1.5B-Instruct | Alibaba, Apache 2.0 | the step 6 model, for reference | 16-bit |
| Phi-3.5-mini-instruct | Microsoft, MIT | a different model family (3.8B) | 16-bit |
| Qwen2.5-7B-Instruct | Alibaba, Apache 2.0 | about 5× larger: does size help? | 4-bit |
| Qwen2.5-Coder-7B-Instruct | Alibaba, Apache 2.0 | trained for code, the closest match to the task | 4-bit |
| Llama-3.1-8B-Instruct | Meta, Llama licence | the most widely used open model | 4-bit |

The 7–8B models are loaded in 4-bit (with `bitsandbytes`) so they fit on a T4. Each model is removed from memory before the next is loaded.

**Llama is gated.** To include it, accept the licence on its Hugging Face page, then add your token in Colab under 🔑 *Secrets* as `HF_TOKEN`, with notebook access switched on. Without a token it is skipped with a message, and the other models still run.

For every model the table shows:
- **AUC** on the 1,500 test issues;
- **within-repository AUC** (repositories with at least 10 sampled test issues);
- coverage, risk and failure rate among delegated issues with CRC at α = 0.10;
- **AUC of main model + LLM**: the LLM score combined with the main model's score on the calibration issues. If this isn't higher than the main model alone, the LLM adds nothing.

The *Main model* and *Repository history* rows should match step 6 (AUC 0.787), which confirms the issues are the same.

Needs only steps 0–2 and a T4 GPU. It takes about 60–75 minutes for all five models; delete lines from `MODELS` to run fewer.

**Output:** `results_llm_comparison.csv`.

In [ ]:
# Compares several open LLMs on the same issues as step 6 (last monthly window, same seed, same prompt).
# Needs steps 0-2 and a T4 GPU. Each model is loaded, used and removed before the next one.
!pip -q install bitsandbytes
import gc, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sklearn.linear_model import LogisticRegression

MODELS = [("Qwen/Qwen2.5-1.5B-Instruct",       "fp16"),
          ("microsoft/Phi-3.5-mini-instruct",  "fp16"),
          ("Qwen/Qwen2.5-7B-Instruct",         "4bit"),
          ("Qwen/Qwen2.5-Coder-7B-Instruct",   "4bit"),
          ("meta-llama/Llama-3.1-8B-Instruct", "4bit")]   # gated: needs an HF token with access, otherwise skipped

# The same issues as step 6: last monthly window (March 2026), 800 calibration and 1,500 test issues, seed 0
t0 = pd.Timestamp("2026-03-01", tz="UTC").value; c1 = t0 - 14 * DAY; c0 = c1 - 30 * DAY
ca_l = np.where((DEC >= c0) & (DEC < c1))[0]
te_l = np.where((DEC >= t0) & (DEC < (pd.Timestamp("2026-04-01", tz="UTC").value)))[0]
_, m_last, X_last, _ = models_store[-1]
pm_l = m_last.predict_proba(X_last)[:, 1]                      # main model of that window
rng_l = np.random.default_rng(0)
cs = rng_l.choice(ca_l, min(800, len(ca_l)), replace=False)
tsx = rng_l.choice(te_l, min(1500, len(te_l)), replace=False)

SYSTEM = "You are an experienced open-source maintainer triaging GitHub issues."
ASK = ("\n\nIf an AI coding agent such as Claude Code, GitHub Copilot or OpenAI Codex were asked to resolve this issue, "
       "is it likely that its pull request would be merged by the maintainers? Answer with Yes or No.")

def score_with(mid, mode, texts, bs=8):
    tok = AutoTokenizer.from_pretrained(mid); tok.padding_side = "left"
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    kw = dict(device_map="cuda")
    if mode == "4bit":
        kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4")
    else:
        kw["dtype"] = torch.float16
    llm = AutoModelForCausalLM.from_pretrained(mid, **kw).eval()
    first = lambda ws: sorted({tok.encode(w, add_special_tokens=False)[0] for w in ws})
    yes, no = first(["Yes", " Yes", "yes"]), first(["No", " No", "no"])
    def prompt(t):
        msg = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": "Here is a GitHub issue.\n\n" + t[:2500] + ASK}]
        return tok.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
    out = []
    with torch.no_grad():
        for i in range(0, len(texts), bs):
            enc = tok([prompt(t) for t in texts[i:i + bs]], return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
            lp = torch.log_softmax(llm(**enc).logits[:, -1, :].float(), -1)
            out += torch.sigmoid(torch.logsumexp(lp[:, yes], -1) - torch.logsumexp(lp[:, no], -1)).cpu().numpy().tolist()
            if i % 800 == 0: print(f"    {i + len(texts[i:i + bs])}/{len(texts)}")
    del llm; gc.collect(); torch.cuda.empty_cache()
    return np.array(out)

def wr_auc(p, y, reps, min_n=10):
    a = [roc_auc_score(y[ix], p[ix]) for ix in pd.Series(np.arange(len(y))).groupby(reps).indices.values()
         if len(ix) >= min_n and 0 < y[ix].mean() < 1]
    return (np.mean(a), len(a)) if a else (np.nan, 0)

texts_c, texts_t = main.text.values[cs].tolist(), main.text.values[tsx].tolist()
yc, yt, rt = Y[cs], Y[tsx], main.repo_id.values[tsx]
SCORES = {"Main model": (pm_l[cs], pm_l[tsx]),
          "Repository history": (REPO_PRIOR[cs] + 1e-6 * rng_l.random(len(cs)), REPO_PRIOR[tsx] + 1e-6 * rng_l.random(len(tsx)))}
for mid, mode in MODELS:
    name = mid.split("/")[-1]
    print(f"\n{name} ({mode})")
    try:
        SCORES[name] = (score_with(mid, mode, texts_c), score_with(mid, mode, texts_t))
    except Exception as e:
        print(f"  skipped: {type(e).__name__}: {str(e)[:200]}"); gc.collect(); torch.cuda.empty_cache()

rows = []
for name, (pc, pt) in SCORES.items():
    lam = lam_crc(pc, yc, ALPHA); wr, nr = wr_auc(pt, yt, rt)
    r = dict(model=name, auc=roc_auc_score(yt, pt), within_repo_auc=wr, repos=nr, **stats(pt, yt, lam))
    if name not in ("Main model", "Repository history"):
        # Does the LLM add anything to the main model? Combine the two scores on the calibration issues.
        lr = LogisticRegression().fit(np.column_stack([pc, pm_l[cs]]), yc)
        r["auc_main_plus_llm"] = roc_auc_score(yt, lr.predict_proba(np.column_stack([pt, pm_l[tsx]]))[:, 1])
    rows.append(r)
LLMS = pd.DataFrame(rows).set_index("model")
print(f"\nSame {len(tsx)} test issues for every model (calibration {len(cs)}), CRC at alpha = {ALPHA}:\n", LLMS.round(3).to_string())
LLMS.to_csv("results_llm_comparison.csv")


## Step 7: Sample for manual label checking
Our label treats every unmerged PR as a failure. But PRs also get closed for reasons that have nothing to do with the agent, such as duplicates, abandoned work, or a feature that is no longer wanted.

This cell draws **200 failed PRs** from the main set, stratified by agent with a fixed seed so the sample is always the same. It saves them to `label_validation_sample.csv` with an empty `closure_reason` column to fill in by hand. The categories are:

`AGENT_FAILURE` · `DUPLICATE_OR_SUPERSEDED` · `ABANDONED_OR_STALE` · `NOT_WANTED_OR_OUT_OF_SCOPE` · `STILL_OPEN_OR_LATE_MERGE` · `OTHER`

Keep your coded copy under a different file name so it never gets overwritten.

In [ ]:
# Open each link, read why the PR was closed, and fill in closure_reason.
mv = pd.read_parquet("routing_main_v2.parquet")
rej = mv[mv.label == 0]
parts = [g.sample(min(len(g), max(10, int(round(200 * len(g) / len(rej))))), random_state=0) for _, g in rej.groupby("agent")]
samp = pd.concat(parts).sample(frac=1, random_state=0).head(200)
sheet = samp[["pr_id", "agent", "html_url", "repo_url", "pr_created"]].copy()
sheet["closure_reason"] = ""   # one of: AGENT_FAILURE | DUPLICATE_OR_SUPERSEDED | ABANDONED_OR_STALE | NOT_WANTED_OR_OUT_OF_SCOPE | STILL_OPEN_OR_LATE_MERGE | OTHER
sheet["notes"] = ""
sheet.to_csv("label_validation_sample.csv", index=False)
print(f"Saved {len(sheet)} rejected PRs to label_validation_sample.csv"); print(sheet.agent.value_counts().to_string())

## Step 8: Pooled vs within-repository AUC of each feature
Computes two AUC values for every single feature on the test months:
- **Pooled AUC:** all repositories mixed together.
- **Within-repository AUC:** averaged over repositories with at least 20 test issues.

If a feature has a high pooled AUC but sits near 0.5 within repositories, it is mostly telling projects apart rather than issues. Saved to `results_feature_within_repo.csv`, which step 13 uses for the pooled-vs-within figure.

In [ ]:
FEATS = ["repo prior", "repo history size", "repo prior (90 days)", "text length", "title length", "code blocks",
         "stack trace", "links", "checklist items", "reproduction steps", "file paths", "bot author", "text score (TF-IDF)"]
allX = np.vstack([Xw[tew] for _, _, Xw, tew in models_store]); allte = np.concatenate([tew for *_, tew in models_store])
yy, rp = Y[allte], main.repo_id.values[allte]; rows_f = []
for j, f in enumerate(FEATS):
    aucs, ws = [], []
    for r in np.unique(rp):
        ix = rp == r
        if ix.sum() >= 20 and len(np.unique(yy[ix])) == 2 and np.ptp(allX[ix, j]) > 0:
            aucs.append(roc_auc_score(yy[ix], allX[ix, j])); ws.append(ix.sum())
    rows_f.append(dict(feature=f, pooled_auc=roc_auc_score(yy, allX[:, j]), within_repo_auc=np.average(aucs, weights=ws) if aucs else np.nan, repos=len(aucs)))
F = pd.DataFrame(rows_f).sort_values("pooled_auc", ascending=False); print(F.round(3).to_string(index=False))
F.to_csv("results_feature_within_repo.csv", index=False)

## Step 9: Results across risk budgets
Repeats the monthly threshold choice for α = 0.05, 0.10, 0.15, 0.20, 0.25 and 0.30 with both CRC and LTT, reusing the models from step 2. For each budget it reports:
- average coverage;
- average and worst-month realised risk;
- how many months went over budget.

The random-screening coverage at the same risk is shown for comparison. The two-panel plot (coverage and realised risk) is saved as `fig_alpha_sweep.png`.

In [ ]:
import matplotlib.pyplot as plt
ALPHAS8 = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]; rows8 = []
for win, mdl, Xw, tew in models_store:
    t0i = pd.Timestamp(win + "-01", tz="UTC").value; c1 = t0i - 14 * DAY; c0 = c1 - 30 * DAY
    caw = np.where((DEC >= c0) & (DEC < c1))[0]; p = mdl.predict_proba(Xw)[:, 1]
    for a in ALPHAS8:
        for meth, f in [("CRC", lam_crc), ("LTT", lam_ltt)]:
            rows8.append(dict(window=win, alpha=a, method=meth, random_cov=min(1, a / (1 - Y[caw].mean())), **stats(p[tew], Y[tew], f(p[caw], Y[caw], a))))
A8 = pd.DataFrame(rows8); A8["violated"] = A8.risk > A8.alpha
summ8 = A8.groupby(["method", "alpha"]).agg(coverage=("coverage", "mean"), risk=("risk", "mean"), max_risk=("risk", "max"),
                                            months_violated=("violated", "sum"), random_cov=("random_cov", "mean")).round(3)
print(summ8.to_string()); A8.to_csv("results_alpha_sweep.csv", index=False)
fig, axs = plt.subplots(1, 2, figsize=(8.5, 3.2))
for meth, mk in [("CRC", "o"), ("LTT", "s")]:
    d = summ8.loc[meth]
    axs[0].plot(d.index, d.coverage, marker=mk, label=f"Main model, {meth}")
    axs[1].plot(d.index, d.risk, marker=mk, label=f"Mean, {meth}"); axs[1].plot(d.index, d.max_risk, marker=mk, ls=":", label=f"Worst month, {meth}")
axs[0].plot(summ8.loc["CRC"].index, summ8.loc["CRC"].random_cov, "k--", lw=1, label="Random screening")
axs[1].plot(ALPHAS8, ALPHAS8, "k--", lw=1, label="Budget (risk = α)")
axs[0].set_title("(a) Share of issues delegated"); axs[1].set_title("(b) Realised risk")
for ax in axs: ax.set_xlabel("Risk budget α"); ax.grid(alpha=.3); ax.legend(fontsize=7)
plt.tight_layout(); plt.savefig("fig_alpha_sweep.png", dpi=300, bbox_inches="tight"); plt.show()

## Step 10: Bi-weekly deployment and alternative policies
Re-runs the deployment with the model retrained **every two weeks** instead of monthly, from September 2025 to March 2026. This gives roughly twice as many deployment windows to judge how often the budget holds.

Policies compared at α = 0.10:

| Policy | Idea |
|---|---|
| Main model, CRC / LTT | Same as step 2, bi-weekly |
| CRC with margin | Calibrate at 0.08 to leave a buffer under the 0.10 budget |
| Online CRC | Adjust the calibration target each window, based on the realised risk two windows earlier (once those outcomes are fully known) |
| Repository-only model | LightGBM on the three repository features only |
| 90-day repository prior | Just the repository's recent success rate |

For each window it also gives a 95% repository-bootstrap interval for the realised risk. The **cold-start** table splits the decisions by how much history the repository had: none, 1–10, 11–50, or more than 50 past outcomes.

**Output:** `results_biweekly.csv`, `results_biweekly_summary.csv`, `results_cold_start.csv`, `fig_biweekly_risk.png`.

This step also keeps each window's training, calibration and test sets and the main model's scores in `BW_STORE`, so steps 14–17 can reuse them without retraining.

In [ ]:
import matplotlib.pyplot as plt
ALPHA, GAMMA, H_DAYS = 0.10, 0.5, 14
starts9 = list(pd.date_range("2025-09-01", "2026-04-01", freq="SMS", tz="UTC"))
REPO3 = np.column_stack([REPO_PRIOR, REPO_N, REPO_PRIOR90])
rng9 = np.random.default_rng(9)
rows9, recs9, a_online, online_risk = [], [], [], []
BW_STORE = []                                                         # kept for steps 14-17
for w in range(len(starts9) - 1):
    t0i, t1i = starts9[w].value, starts9[w + 1].value
    c1 = t0i - H_DAYS * DAY; c0 = c1 - 30 * DAY
    tr = np.where(DEC < c0)[0]; ca = np.where((DEC >= c0) & (DEC < c1))[0]; te = np.where((DEC >= t0i) & (DEC < t1i))[0]
    if len(te) < 50 or len(ca) < 50: continue
    other = np.concatenate([ca, te])
    TXT = tfidf_scores(tr, other); X = feat_main(TXT)
    pm9 = lgbm(X, tr).predict_proba(X)[:, 1]
    BW_STORE.append(dict(window=starts9[w].strftime("%Y-%m-%d"), tr=tr, ca=ca, te=te, TXT=TXT, pm=pm9))
    pr9 = lgbm(REPO3, tr).predict_proba(REPO3)[:, 1]
    # online target: start at ALPHA; after two windows, adjust with the realised risk that is now fully observed
    k = len(a_online)
    a_t = ALPHA if k < 2 else float(np.clip(a_online[-1] + GAMMA * (ALPHA - online_risk[k - 2]), 0.01, 0.20))
    pols = {"Main model, CRC": (pm9, lam_crc(pm9[ca], Y[ca], ALPHA)),
            "Main model, LTT": (pm9, lam_ltt(pm9[ca], Y[ca], ALPHA)),
            "Main model, CRC with margin (0.08)": (pm9, lam_crc(pm9[ca], Y[ca], 0.08)),
            "Main model, online CRC": (pm9, lam_crc(pm9[ca], Y[ca], a_t)),
            "Repository-only model (retrained), CRC": (pr9, lam_crc(pr9[ca], Y[ca], ALPHA)),
            "90-day repository prior, CRC": (REPO_PRIOR90 + 1e-6 * rng9.random(len(main)), None)}
    p90 = pols["90-day repository prior, CRC"][0]; pols["90-day repository prior, CRC"] = (p90, lam_crc(p90[ca], Y[ca], ALPHA))
    for name, (p, lam) in pols.items():
        s = stats(p[te], Y[te], lam)
        # per-window bootstrap over repositories for the realised risk
        reps = main.repo_id.values[te]; ur = np.unique(reps); idx_by = {r: np.where(reps == r)[0] for r in ur}; bs = []
        for _ in range(300):
            ix = np.concatenate([idx_by[r] for r in rng9.choice(ur, len(ur))]); bs.append(np.mean((p[te][ix] >= lam) & (Y[te][ix] == 0)))
        rows9.append(dict(window=starts9[w].strftime("%Y-%m-%d"), policy=name, n_test=len(te), auc=roc_auc_score(Y[te], p[te]),
                          calib_target=a_t if "online" in name else (0.08 if "margin" in name else ALPHA),
                          risk_ci_low=np.quantile(bs, .025), risk_ci_high=np.quantile(bs, .975), **s))
    a_online.append(a_t); online_risk.append(stats(pm9[te], Y[te], pols["Main model, online CRC"][1])["risk"])
    lm = pols["Main model, CRC"][1]
    recs9.append(pd.DataFrame(dict(window=starts9[w].strftime("%Y-%m-%d"), y=Y[te], delegated=pm9[te] >= lm, repo_hist=REPO_HIST[te])))
bw = pd.DataFrame(rows9); bw["violated"] = bw.risk > ALPHA
summ9 = bw.groupby("policy").agg(windows=("risk", "size"), mean_auc=("auc", "mean"), coverage=("coverage", "mean"), risk=("risk", "mean"),
         max_risk=("risk", "max"), windows_violated=("violated", "sum"), fail_among_delegated=("cond_fail", "mean")).round(3).sort_values("risk")
print("Bi-weekly rolling deployment, alpha = 0.10:\n", summ9.to_string())
print("\nPer window, main model CRC (with 95% repository-bootstrap CI of realised risk):\n",
      bw[bw.policy == "Main model, CRC"][["window", "n_test", "coverage", "risk", "risk_ci_low", "risk_ci_high"]].round(3).to_string(index=False))
print("\nOnline calibration targets by window:", [round(a, 3) for a in a_online])
REC9 = pd.concat(recs9, ignore_index=True)
REC9["history"] = pd.cut(REC9.repo_hist, [-1, 0, 10, 50, np.inf], labels=["new (0)", "1-10", "11-50", ">50"])
cold = REC9.groupby("history").apply(lambda g: pd.Series(dict(n=len(g), success_rate=g.y.mean(), coverage=g.delegated.mean(),
        risk=np.mean(g.delegated & (g.y == 0)), fail_among_delegated=(1 - g.y[g.delegated].mean()) if g.delegated.any() else np.nan)))
print("\nCold start: main-model CRC decisions by repository history:\n", cold.round(3).to_string())
bw.to_csv("results_biweekly.csv", index=False); summ9.to_csv("results_biweekly_summary.csv"); cold.to_csv("results_cold_start.csv")
fig, ax = plt.subplots(figsize=(7, 3.2))
for name, c, ls in [("Main model, CRC", "#2ca02c", "-"), ("Main model, CRC with margin (0.08)", "#9467bd", "-"), ("Main model, online CRC", "#1f77b4", "--"),
                    ("Repository-only model (retrained), CRC", "#ff7f0e", ":"), ("90-day repository prior, CRC", "#d62728", "-.")]:
    d = bw[bw.policy == name]; ax.plot(d.window, d.risk, marker="o", ms=3, color=c, ls=ls, label=name)
d = bw[bw.policy == "Main model, CRC"]; ax.fill_between(d.window, d.risk_ci_low, d.risk_ci_high, color="#2ca02c", alpha=.15)
ax.axhline(ALPHA, color="k", ls="--", lw=.8); ax.set_ylabel("Realised risk"); ax.tick_params(axis="x", rotation=60, labelsize=7); ax.grid(alpha=.3)
ax.legend(fontsize=6.5, loc="upper center", bbox_to_anchor=(.5, -.45), ncol=2, frameon=False)
plt.savefig("fig_biweekly_risk.png", dpi=300, bbox_inches="tight"); plt.show()

## Step 11: Look at the PR timeline table
We use the PR creation time as the moment the agent was started. That is an approximation, because the agent may have been assigned earlier. AIDev has a `pr_timeline` table of PR events. This cell only prints its columns, a few rows, and the most common event types, to see whether an earlier start time (for example an assignment event) can be recovered. It doesn't change any results.

In [ ]:
# Only reads the schema and the first row group, the full table is large.
import pyarrow.parquet as pq
from huggingface_hub import hf_hub_download
pf = pq.ParquetFile(hf_hub_download("hao-li/AIDev", "pr_timeline.parquet", repo_type="dataset"))
print("Rows:", pf.metadata.num_rows, "| row groups:", pf.num_row_groups)
print("Columns:", pf.schema_arrow.names)
sample = pf.read_row_group(0).to_pandas()
print(sample.head(5).to_string()[:3000])
for col in [c for c in sample.columns if "event" in c.lower() or "type" in c.lower()]:
    print(f"\nMost common values of '{col}' in the first row group:\n", sample[col].value_counts().head(25).to_string())

## Step 12: Accuracy, precision, recall and F1
Treats the router as a classifier where **"delegate"** is the positive class:

| | Merged in 14 days | Not merged |
|---|---|---|
| **Delegated** | true positive | false positive (a failed agent PR) |
| **Not delegated** | false negative | true negative |

Reports accuracy, balanced accuracy, precision, recall, F1, coverage and risk. It compares the CRC router (α = 0.10) with simply delegating everything, both overall and per month. Note that precision = 1 − failure rate among delegated, and risk = false positives ÷ all issues.

**Output:** `results_classification_metrics.csv` and `results_classification_by_month.csv`.

In [ ]:
# Positive class = "delegate". A delegated issue that merged is a true positive.
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             balanced_accuracy_score, confusion_matrix)
REC11 = pd.read_csv("results_rolling_predictions.csv")
y11, d11 = REC11.y.values.astype(int), REC11.delegated.astype(int).values
def metrics(y_, pred):
    tn, fp, fn, tp = confusion_matrix(y_, pred, labels=[0, 1]).ravel()
    return dict(accuracy=accuracy_score(y_, pred), balanced_accuracy=balanced_accuracy_score(y_, pred),
                precision=precision_score(y_, pred, zero_division=0), recall=recall_score(y_, pred, zero_division=0),
                f1=f1_score(y_, pred, zero_division=0), coverage=pred.mean(), risk=fp / len(y_),
                TP=tp, FP=fp, FN=fn, TN=tn)
M11 = pd.DataFrame({"Router (CRC, alpha=0.10)": metrics(y11, d11),
                    "Delegate everything": metrics(y11, np.ones_like(y11))}).T
print(M11.round(3).to_string())
per_win11 = REC11.groupby("window").apply(lambda g: pd.Series(metrics(g.y.values.astype(int), g.delegated.astype(int).values)))
print("\nPer month:\n", per_win11[["accuracy", "precision", "recall", "f1", "coverage", "risk"]].round(3).to_string())
M11.to_csv("results_classification_metrics.csv"); per_win11.to_csv("results_classification_by_month.csv")

## Step 13: Thesis figures
Draws every figure in the thesis from the saved result files rather than from typed-in numbers. The cell checks first that the files it needs exist.

| File | Figure |
|---|---|
| `fig_censoring.png` | Share of PRs still open, by month |
| `fig_pipeline.png` | Pipeline diagram |
| `fig_pooled_vs_within.png` | Pooled vs within-repository AUC |
| `fig_shap.png` | Mean SHAP per feature |
| `fig_by_history.png` | AUC by amount of repository history |
| `fig_drift.png` | Agent success rates before and in 2026 |
| `fig_rolling_risk.png` | Realised risk per month |
| `fig_calib_size.png` | Realised risk vs calibration size |

It can also run in a fresh session if you upload the `results_*.csv` files and `routing_main_v2.parquet` first.

In [ ]:
# Rebuilds every thesis figure from the saved results_*.csv files.
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Patch
from huggingface_hub import hf_hub_download
plt.rcParams.update({"font.size": 9, "font.family": "serif", "savefig.bbox": "tight"})
need = ["results_rolling_windows.csv", "results_rolling_calib_size.csv", "results_bootstrap_ci.csv", "results_by_repo_history.csv",
        "results_shap.csv", "results_feature_within_repo.csv", "routing_main_v2.parquet"]
missing = [f for f in need if not os.path.exists(f)]
assert not missing, f"Missing files: {missing}. Run the earlier steps first (results_feature_within_repo.csv comes from step 8)."

# Fig 3.1: censoring — share of PRs still open at the snapshot, by month
pr = pd.read_parquet(hf_hub_download("hao-li/AIDev", "pull_request.parquet", repo_type="dataset"), columns=["id", "created_at", "merged_at", "closed_at"]).drop_duplicates("id")
for c in ["created_at", "merged_at", "closed_at"]: pr[c] = pd.to_datetime(pr[c], utc=True, errors="coerce")
pr["month"] = pr.created_at.dt.strftime("%Y-%m"); pr["open"] = pr.merged_at.isna() & pr.closed_at.isna()
cm = pr.groupby("month").agg(prs=("id", "size"), open_share=("open", "mean")).tail(12)
fig, ax = plt.subplots(figsize=(6, 2.8)); ax2 = ax.twinx()
ax2.bar(range(len(cm)), cm.prs / 1000, color="#cfd8e3", label="PRs created (thousands)")
ax.plot(range(len(cm)), cm.open_share, "o-", color="#d62728", label="Share still open at snapshot")
ax.set_zorder(ax2.get_zorder() + 1); ax.patch.set_visible(False)
ax.set_xticks(range(len(cm))); ax.set_xticklabels([m[2:] for m in cm.index], rotation=45); ax.set_ylabel("Share open"); ax2.set_ylabel("PRs (thousands)")
h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels(); ax.legend(h1 + h2, l1 + l2, fontsize=7.5, loc="upper left"); ax.grid(alpha=.3)
plt.savefig("fig_censoring.png", dpi=300); plt.show()

# Fig 4.1: pipeline diagram
fig, ax = plt.subplots(figsize=(10, 2.9)); ax.set_xlim(0, 11.4); ax.set_ylim(0, 2.9); ax.axis("off")
def box(x, y, w, h, t, fc):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.05", fc=fc, ec="#333", lw=1)); ax.text(x + w / 2, y + h / 2, t, ha="center", va="center", fontsize=8)
def arr(a, b, c, d): ax.annotate("", xy=(c, d), xytext=(a, b), arrowprops=dict(arrowstyle="->", lw=1))
box(0.1, 1.1, 1.5, 0.75, "Issue considered\nfor delegation\n(time t)", "#eef3fb")
for y, t in zip([2.05, 1.15, 0.25], ["Repository history\n(outcomes known before t)", "Issue structure\n(9 features)", "Issue text\n(TF-IDF score)"]):
    box(2.1, y, 2.4, 0.62, t, "#fff4e6"); arr(1.65, 1.47, 2.05, y + .31); arr(4.55, y + .31, 4.95, 1.47)
box(5.0, 1.1, 1.7, 0.75, "LightGBM\nretrained monthly\n$\\hat p$(success)", "#eaf6ea"); arr(6.75, 1.47, 7.15, 1.47)
box(7.2, 1.1, 2.2, 0.75, "Threshold $\\hat\\lambda$ (CRC / LTT)\non recent, fully\nobserved outcomes", "#f3eafa")
box(10.05, 1.95, 1.25, 0.6, "Delegate\nto agent", "#e8f5e9"); box(10.05, 0.35, 1.25, 0.6, "Do not\ndelegate", "#fdecea")
arr(9.45, 1.65, 10.0, 2.25); arr(9.45, 1.3, 10.0, 0.65)
ax.text(9.6, 2.15, "$\\hat p\\geq\\hat\\lambda$", fontsize=8.5, ha="center"); ax.text(9.6, 0.72, "$\\hat p<\\hat\\lambda$", fontsize=8.5, ha="center")
plt.savefig("fig_pipeline.png", dpi=300); plt.show()

# Fig 5.1: pooled vs within-repository AUC (main model + every feature)
F = pd.read_csv("results_feature_within_repo.csv"); CI = pd.read_csv("results_bootstrap_ci.csv", index_col=0)
wr_main = CI.loc["within_repo_auc_main", "estimate"] if "within_repo_auc_main" in CI.index else np.nan
rows = [("Main model (stacked)", CI.loc["auc_main", "estimate"], wr_main)]
rows += [(r.feature[0].upper() + r.feature[1:], r.pooled_auc, r.within_repo_auc) for r in F.itertuples() if r.feature != "bot author"]
fig, ax = plt.subplots(figsize=(5.2, 3.6)); y = np.arange(len(rows))[::-1]
for yi, (n, p, w) in zip(y, rows): ax.plot([w, p], [yi, yi], color="lightgray", lw=1.5, zorder=1)
ax.scatter([r[1] for r in rows], y, s=18, color="#1f77b4", label="Pooled across repositories", zorder=2)
ax.scatter([r[2] for r in rows], y, s=18, color="#d62728", marker="D", label="Within repositories", zorder=3)
ax.axvline(.5, color="k", lw=.8, ls="--"); ax.set_yticks(y); ax.set_yticklabels([r[0] for r in rows])
ax.set_xlabel("ROC-AUC (rolling test windows)"); ax.set_xlim(.40, .80); ax.grid(axis="x", alpha=.3); ax.legend(fontsize=7, loc="lower right")
plt.savefig("fig_pooled_vs_within.png", dpi=300); plt.show()

# Fig 5.2: SHAP — mean |SHAP| averaged over the monthly models
SH = pd.read_csv("results_shap.csv", index_col=0)["mean"].sort_values()
REP = {"repo prior", "repo history size", "repo prior (90 days)"}
fig, ax = plt.subplots(figsize=(5.5, 3.2))
ax.barh(range(len(SH)), SH.values, color=["#1f77b4" if f in REP else "#ff7f0e" for f in SH.index])
ax.set_yticks(range(len(SH))); ax.set_yticklabels(SH.index); ax.set_xlabel("Mean |SHAP| (average over monthly models)")
ax.legend(handles=[Patch(color="#1f77b4", label="Repository features"), Patch(color="#ff7f0e", label="Issue features")], fontsize=7.5, loc="lower right"); ax.grid(axis="x", alpha=.3)
plt.savefig("fig_shap.png", dpi=300); plt.show()

# Fig 5.3: AUC by repository history
BH = pd.read_csv("results_by_repo_history.csv", index_col=0)
fig, ax = plt.subplots(figsize=(4.8, 2.8)); x = np.arange(len(BH)); w = .38
ax.bar(x - w / 2, BH.auc_main, w, label="Main model"); ax.bar(x + w / 2, BH.auc_repo, w, label="Repository history")
for i, n in enumerate(BH.n): ax.text(i, 0.95, f"n={int(n):,}", ha="center", fontsize=7)
ax.axhline(.5, color="k", ls="--", lw=.8); ax.set_ylim(.4, 1.0); ax.set_xticks(x); ax.set_xticklabels(BH.index)
ax.set_xlabel("Past outcomes known for the repository"); ax.set_ylabel("Pooled ROC-AUC"); ax.legend(fontsize=7.5, loc="lower right"); ax.grid(axis="y", alpha=.3)
plt.savefig("fig_by_history.png", dpi=300); plt.show()

# Fig 5.4: agent drift within repositories (same computation as step 5)
mv = pd.read_parquet("routing_main_v2.parquet")
mv["period"] = np.where(pd.to_datetime(mv.pr_created, utc=True) < pd.Timestamp("2026-01-01", tz="UTC"), "before", "2026")
names = {"Claude_Code": "Claude Code", "Copilot": "Copilot", "OpenAI_Codex": "Codex"}; rows_d = []
for ag in names:
    g = mv[mv.agent == ag]; both = g.groupby("repo_id").period.nunique(); gb = g[g.repo_id.isin(both[both == 2].index)]
    raw = g.groupby("period").label.mean(); same = gb.groupby("period").label.mean()
    wtd = gb.groupby(["period", "repo_id"]).label.mean().groupby("period").mean()
    rows_d.append((names[ag], raw, same, wtd))
fig, axs = plt.subplots(1, 3, figsize=(8, 2.6), sharey=True)
for k, title in enumerate(["All repositories", "Same repositories, pooled", "Same repositories, repo-weighted"]):
    ax = axs[k]; xx = np.arange(3)
    ax.bar(xx - .19, [r[k + 1].get("before") for r in rows_d], .38, label="Before 2026")
    ax.bar(xx + .19, [r[k + 1].get("2026") for r in rows_d], .38, label="2026")
    ax.set_xticks(xx); ax.set_xticklabels([r[0] for r in rows_d], fontsize=7.5); ax.set_title(title, fontsize=8.5); ax.grid(axis="y", alpha=.3)
axs[0].set_ylabel("Success rate (merged ≤ 14 days)"); axs[0].legend(fontsize=7); axs[0].set_ylim(0, 1)
plt.savefig("fig_drift.png", dpi=300); plt.show()

# Fig 5.5: realised risk per month
RW = pd.read_csv("results_rolling_windows.csv")
series = [("Main: retrained + recalibrated", "CRC", "Main model, CRC", "#2ca02c", "o", "-"),
          ("Main: retrained + recalibrated", "LTT", "Main model, LTT", "#1f77b4", "s", "-"),
          ("Baseline: repo history only", "CRC", "Repository history, CRC", "#d62728", "^", "--"),
          ("Baseline: random", "CRC", "Random scores, CRC", "gray", "x", ":")]
fig, ax = plt.subplots(figsize=(4.6, 3.0))
for pol, meth, lab, c, m, ls in series:
    d = RW[(RW.policy == pol) & (RW.method == meth)].sort_values("window")
    ax.plot(d.window, d.risk, marker=m, ms=3.5, ls=ls, color=c, lw=1.1, label=lab)
ax.axhline(.10, color="k", lw=.9, ls="--"); ax.text(-.15, .104, "budget α = 0.10", fontsize=7, ha="left", va="bottom")
ax.set_ylabel("Realised risk"); ax.set_ylim(0, max(0.28, RW.risk.max() + 0.02)); ax.set_xlabel("Deployment month"); ax.tick_params(axis="x", rotation=45); ax.grid(alpha=.3)
ax.legend(fontsize=6.8, loc="upper center", bbox_to_anchor=(.5, -.32), ncol=2, frameon=False)
plt.savefig("fig_rolling_risk.png", dpi=300); plt.show()

# Fig 5.6: chronological calibration size
S = pd.read_csv("results_rolling_calib_size.csv")
agg = S.groupby(["n_calib", "method"]).agg(risk=("risk", "mean"), viol=("violated", "sum"), w=("risk", "size")).reset_index()
fig, ax = plt.subplots(figsize=(4.8, 2.8))
for meth, mk, off in [("CRC", "o", 6), ("LTT", "s", -12)]:
    d = agg[agg.method == meth]; ax.plot(d.n_calib, d.risk, marker=mk, label=meth if meth == "CRC" else "LTT (δ = 0.1)")
    for a, b, v, w_ in zip(d.n_calib, d.risk, d.viol, d.w): ax.annotate(f"{int(v)}/{int(w_)}", (a, b), textcoords="offset points", xytext=(0, off), ha="center", fontsize=7)
ax.axhline(.1, color="k", ls="--", lw=.8); ax.set_xscale("log"); ns = sorted(agg.n_calib.unique()); ax.set_xticks(ns); ax.set_xticklabels(ns); ax.minorticks_off()
ax.set_xlabel("Calibration set size (most recent issues)"); ax.set_ylabel("Mean realised risk"); ax.legend(fontsize=7.5); ax.grid(alpha=.3)
plt.savefig("fig_calib_size.png", dpi=300); plt.show()

figs = ["fig_censoring.png", "fig_pipeline.png", "fig_pooled_vs_within.png", "fig_shap.png", "fig_by_history.png", "fig_drift.png", "fig_rolling_risk.png", "fig_calib_size.png"]
print("Saved:", ", ".join(figs))

## Step 14: Warm-up policy for new repositories
Step 10 showed that for repositories with **no agent history**, screening gives no protection: the issues it delegates fail about as often as if we hadn't screened at all. The model knows nothing about these projects yet.

This step compares four ways of handling repositories with fewer than *k* known outcomes (*k* = 1, 3, 5):

| Policy | What happens to an issue in a repository with fewer than *k* known outcomes |
|---|---|
| Screen everything (current) | Treated like any other issue |
| Hold back | Never delegated until *k* outcomes are known |
| Warm-up | Always delegated, so the team learns about the project quickly |
| Trial | A brand-new repository's first *k* issues are delegated as a trial; its later issues wait until an outcome is known, then are screened normally |

The budget stays at 10% of **all** issues. For the warm-up and trial policies, the failures of the forced issues are counted first, and the threshold for everything else is chosen so the total stays within budget. In other words, learning about new projects is paid for out of the same budget.

The output shows coverage, risk and violated windows for each policy, plus results split by repository history. `new_repos_tried` counts new repositories that got at least one delegated issue. A policy that never tries new repositories can never learn about them.

Note: every issue in the data was actually attempted by an agent, so outcomes are known even for issues a policy holds back. That is what makes this comparison possible.

**Output:** `results_warmup.csv`, `results_warmup_windows.csv`, `results_warmup_by_history.csv`.

In [ ]:
# Uses the bi-weekly models saved by step 10 (BW_STORE), so run steps 0-2 and 10 first.
WARM_K = [1, 3, 5]
hist_bin = lambda h: pd.cut(h, [-1, 0, 10, 50, np.inf], labels=["new (0)", "1-10", "11-50", ">50"])

def lam_crc_forced(p, y, a, forced):
    # CRC threshold when the `forced` issues are delegated no matter what the threshold is.
    # Their failures are counted at every threshold, so the remaining budget shrinks.
    n = len(p)
    k = fails_above(p[~forced], y[~forced]) + np.sum(forced & (y == 0))
    ok = np.where(k / (n + 1) + 1 / (n + 1) <= a)[0]
    return GRID[ok[0]] if len(ok) else 1.01

# Position of each issue among its repository's issues, in time order (0 = the repository's first agent issue).
RANK = main.assign(_d=DEC).groupby("repo_id")._d.rank(method="first").values - 1

def decide(p, hist, rank, lam, rule, k):
    if rule == "screen": return p >= lam
    if rule == "hold":   return (hist >= k) & (p >= lam)      # never delegate until k outcomes are known
    if rule == "warm":   return (hist < k) | (p >= lam)       # always delegate until k outcomes are known
    if rule == "trial":                                       # new repository: delegate only its first k issues
        return np.where(hist == 0, rank < k, p >= lam)

policies = [("Screen everything (current)", "screen", 0)] + \
           [(f"Hold back until {k} known", "hold", k) for k in WARM_K] + \
           [(f"Warm-up: delegate until {k} known", "warm", k) for k in WARM_K] + \
           [(f"Trial: first {k} issue(s) of a new repo", "trial", k) for k in WARM_K]

rows14, recs14 = [], []
for st in BW_STORE:
    ca, te, p = st["ca"], st["te"], st["pm"]
    hc, ht, rc, rt = REPO_HIST[ca], REPO_HIST[te], RANK[ca], RANK[te]
    for name, rule, k in policies:
        if rule == "screen": lam = lam_crc(p[ca], Y[ca], ALPHA)
        elif rule == "hold": lam = lam_crc(np.where(hc >= k, p[ca], -1.0), Y[ca], ALPHA)
        elif rule == "warm": lam = lam_crc_forced(p[ca], Y[ca], ALPHA, hc < k)
        else:                                                 # trial issues are forced, other new-repo issues held back
            lam = lam_crc_forced(np.where(hc == 0, -1.0, p[ca]), Y[ca], ALPHA, (hc == 0) & (rc < k))
        dlg = decide(p[te], ht, rt, lam, rule, k)
        rows14.append(dict(window=st["window"], policy=name, coverage=dlg.mean(), risk=np.mean(dlg & (Y[te] == 0)),
                           cond_fail=1 - Y[te][dlg].mean() if dlg.any() else np.nan,
                           new_repos_tried=len(np.unique(main.repo_id.values[te][dlg & (ht == 0)]))))
        recs14.append(pd.DataFrame(dict(policy=name, y=Y[te], delegated=dlg, history=hist_bin(ht))))

W14 = pd.DataFrame(rows14); W14["violated"] = W14.risk > ALPHA
order = [p[0] for p in policies]
summ14 = (W14.groupby("policy").agg(coverage=("coverage", "mean"), risk=("risk", "mean"), max_risk=("risk", "max"),
                                    windows_violated=("violated", "sum"), fail_among_delegated=("cond_fail", "mean"),
                                    new_repos_tried=("new_repos_tried", "sum"))
          .reindex(order).round(3))
print("Warm-up policies, bi-weekly windows, alpha = 0.10:\n", summ14.to_string())

R14 = pd.concat(recs14, ignore_index=True)
by_hist14 = R14.groupby(["policy", "history"], observed=True).apply(lambda g: pd.Series(dict(
    n=len(g), coverage=g.delegated.mean(), fail_among_delegated=(1 - g.y[g.delegated].mean()) if g.delegated.any() else np.nan)))
print("\nBy repository history:\n", by_hist14.unstack("history").reindex(order).round(3).to_string())
W14.to_csv("results_warmup_windows.csv", index=False); summ14.to_csv("results_warmup.csv"); by_hist14.to_csv("results_warmup_by_history.csv")


## Step 15: Checking the decision time
We use the moment the agent's PR was opened as the moment the team decided to delegate. This step checks how good that approximation is, in two parts.

**Part A: the PR timeline.** It reads four kinds of events from AIDev's `pr_timeline` table: `assigned`, `copilot_work_started`, `ready_for_review` and `convert_to_draft`. For each agent it reports how many PRs have each event and how many hours after PR creation it happened. Copilot opens a draft PR as soon as it is given an issue and marks it *ready for review* when finished. So for Copilot the time between the two shows how long the agent worked. For agents that only open a PR after finishing, that is roughly how much later PR creation is than the real decision.

**Part B: sensitivity test.** It pretends every decision was made 24, 72 or 168 hours earlier than the PR creation time. It then recomputes the repository-history features with only the outcomes known by that earlier moment, retrains, and repeats the bi-weekly evaluation. If AUC, within-repository AUC and risk barely change, the conclusions don't depend on the exact decision time.

**Output:** `results_timeline_lags.csv`, `results_decision_shift.csv`.

In [ ]:
# Part A reads only three columns of the PR timeline, filtered to the few events we need.
import pyarrow.parquet as pq
EVENTS = ["assigned", "copilot_work_started", "ready_for_review", "convert_to_draft"]
tl = pq.read_table(dl("pr_timeline.parquet"), columns=["pr_id", "event", "created_at"],
                   filters=[("event", "in", EVENTS)]).to_pandas()
tl = tl[tl.pr_id.isin(set(main.pr_id))]
tl["created_at"] = pd.to_datetime(tl.created_at, utc=True, errors="coerce")
first_ev = tl.pivot_table(index="pr_id", columns="event", values="created_at", aggfunc="min")

pt = main[["pr_id", "agent", "pr_created"]].set_index("pr_id").join(first_ev, how="left")
lag_rows = []
for ag, g in pt.groupby("agent"):
    r = dict(agent=ag, prs=len(g))
    for ev in EVENTS:
        if ev not in g: continue
        h = (g[ev] - g.pr_created).dt.total_seconds() / 3600
        r[f"{ev}: share"] = h.notna().mean()
        r[f"{ev}: median h"] = h.median()
    lag_rows.append(r)
LAG = pd.DataFrame(lag_rows).set_index("agent")
print("Timeline events after PR creation (share of PRs with the event, median hours from PR creation):\n",
      LAG.round(2).to_string())

# How long Copilot works between opening its draft PR and marking it ready (an upper bound on how
# much later PR creation can be than the decision, for agents that open the PR only when done).
if "ready_for_review" in pt:
    cp = pt[pt.agent == "Copilot"]
    work_h = ((cp.ready_for_review - cp.pr_created).dt.total_seconds() / 3600).dropna()
    print(f"\nCopilot working time, PR opened -> ready for review: median {work_h.median():.2f} h, "
          f"p75 {work_h.quantile(.75):.2f} h, p90 {work_h.quantile(.9):.2f} h (n = {len(work_h):,})")

# Part B: move every decision earlier by `shift` hours and recompute the history features.
# The split into training / calibration / test stays the same; only what counts as "known" changes.
def known_before_at(groups, dec, window_days=None):
    s, c = np.zeros(len(main)), np.zeros(len(main))
    for _, idx in pd.Series(np.arange(len(main))).groupby(np.asarray(groups)).indices.items():
        o = np.argsort(RES[idx]); rs = RES[idx][o]; cy = np.concatenate([[0.0], np.cumsum(Y[idx][o])])
        hi = np.searchsorted(rs, dec[idx], side="left")
        lo = np.zeros_like(hi) if window_days is None else np.searchsorted(rs, dec[idx] - window_days * DAY, side="left")
        s[idx], c[idx] = cy[hi] - cy[lo], hi - lo
    return s, c

def within_repo_auc(p, y, repos, min_n=20):
    aucs = [roc_auc_score(y[ix], p[ix]) for ix in pd.Series(np.arange(len(y))).groupby(repos).indices.values()
            if len(ix) >= min_n and 0 < y[ix].mean() < 1]
    return np.mean(aucs), len(aucs)

rows15 = []
for shift in [0, 24, 72, 168]:
    dec = DEC - shift * 3600 * 10**9
    sg_, cg_ = known_before_at(np.zeros(len(main)), dec); G_ = np.where(cg_ > 0, sg_ / np.maximum(cg_, 1), 0.5)
    sr_, cr_ = known_before_at(main.repo_id.values, dec); sr9_, cr9_ = known_before_at(main.repo_id.values, dec, 90)
    REPO_ = np.column_stack([(sr_ + 10 * G_) / (cr_ + 10), np.log1p(cr_), (sr9_ + 10 * G_) / (cr9_ + 10)])
    P, Yt, Rt = [], [], []
    for st in BW_STORE:
        X_ = np.column_stack([REPO_, ISS, st["TXT"]])
        p = lgbm(X_, st["tr"]).predict_proba(X_)[:, 1]
        ca, te = st["ca"], st["te"]
        s = stats(p[te], Y[te], lam_crc(p[ca], Y[ca], ALPHA))
        rows15.append(dict(shift_hours=shift, window=st["window"], auc=roc_auc_score(Y[te], p[te]), **s))
        P.append(p[te]); Yt.append(Y[te]); Rt.append(main.repo_id.values[te])
    P, Yt, Rt = map(np.concatenate, (P, Yt, Rt))
    wr, nr = within_repo_auc(P, Yt, Rt)
    rows15.append(dict(shift_hours=shift, window="pooled", auc=roc_auc_score(Yt, P), within_repo_auc=wr, repos=nr))

T15 = pd.DataFrame(rows15)
w15 = T15[T15.window != "pooled"].assign(violated=lambda d: d.risk > ALPHA)
summ15 = (w15.groupby("shift_hours").agg(coverage=("coverage", "mean"), risk=("risk", "mean"), max_risk=("risk", "max"),
                                         windows_violated=("violated", "sum"))
          .join(T15[T15.window == "pooled"].set_index("shift_hours")[["auc", "within_repo_auc", "repos"]]).round(3))
print("\nDecision moved earlier by N hours (bi-weekly windows, alpha = 0.10):\n", summ15.to_string())
LAG.to_csv("results_timeline_lags.csv"); summ15.to_csv("results_decision_shift.csv")

## Step 16: Separate thresholds per group (hierarchical risk control)
One global threshold keeps the *overall* risk near budget but can leave some agents, project types or repositories far over it. This step gives groups their own CRC threshold and checks whether that spreads the protection more evenly.

| Scheme | Groups | If a group is too small |
|---|---|---|
| Global threshold | none | – |
| By repository history | new, 1–10, 11–50, more than 50 known outcomes | use the global threshold |
| By agent | Claude Code, Copilot, Codex, … | use the global threshold |
| By agent × history | each agent within each history group | use that agent's threshold |
| By repository | each repository | use its history group's threshold |

A group gets its own threshold only if it has at least 100 calibration issues (`MIN_CAL`). Smaller groups borrow the threshold of the larger group above them, which is what makes it *hierarchical*.

Besides overall coverage and risk, the output shows how often groups end up over budget: agent-windows, history-group-windows and repository-windows (repositories with at least 20 test issues). It also shows coverage and failure rate for new repositories.

**Output:** `results_hierarchical.csv`, `results_hierarchical_windows.csv`.

In [ ]:
# Uses BW_STORE from step 10 and hist_bin from step 14.
MIN_CAL = 100                                                  # smallest calibration group that gets its own threshold
AGENT = main.agent.values
HBIN = hist_bin(REPO_HIST).astype(str)
REPO = main.repo_id.values

def group_thresholds(p, ca, keys, parent):
    # One CRC threshold per group; groups with fewer than MIN_CAL calibration issues use their parent's threshold.
    lam = {}
    for g, pos in pd.Series(np.arange(len(ca))).groupby(keys[ca]).indices.items():
        ix = ca[pos]
        if len(ix) >= MIN_CAL: lam[g] = lam_crc(p[ix], Y[ix], ALPHA)
    return np.array([lam.get(g, par) for g, par in zip(keys, parent)])

def schemes(p, ca):
    glob = np.full(len(main), lam_crc(p[ca], Y[ca], ALPHA))
    by_hist = group_thresholds(p, ca, HBIN, glob)
    by_agent = group_thresholds(p, ca, AGENT, glob)
    ag_hist = group_thresholds(p, ca, np.char.add(AGENT.astype(str), "|" + HBIN), by_agent)   # agent x history, falls back to agent
    by_repo = group_thresholds(p, ca, REPO, by_hist)                                           # repo, falls back to history group
    return {"Global threshold": glob, "By repository history": by_hist, "By agent": by_agent,
            "By agent x history (fallback: agent)": ag_hist, "By repository (fallback: history group)": by_repo}

def violations(dlg, y, keys, min_n=20):
    # Share of groups (with >= min_n test issues) whose realised risk is above the budget.
    r = [np.mean(dlg[ix] & (y[ix] == 0)) for ix in pd.Series(np.arange(len(y))).groupby(keys).indices.values() if len(ix) >= min_n]
    return np.mean(np.array(r) > ALPHA), len(r)

rows16 = []
for st in BW_STORE:
    p, ca, te = st["pm"], st["ca"], st["te"]
    for name, lam in schemes(p, ca).items():
        dlg, y = p[te] >= lam[te], Y[te]
        va, na = violations(dlg, y, AGENT[te]); vh, nh = violations(dlg, y, HBIN[te]); vr, nr = violations(dlg, y, REPO[te])
        rows16.append(dict(window=st["window"], scheme=name, coverage=dlg.mean(), risk=np.mean(dlg & (y == 0)),
                           cond_fail=1 - y[dlg].mean() if dlg.any() else np.nan,
                           agent_viol=va, n_agent=na, hist_viol=vh, n_hist=nh, repo_viol=vr, n_repo=nr,
                           new_repo_cov=dlg[HBIN[te] == "new (0)"].mean(),
                           new_repo_fail=1 - y[dlg & (HBIN[te] == "new (0)")].mean() if (dlg & (HBIN[te] == "new (0)")).any() else np.nan))

W16 = pd.DataFrame(rows16); W16["violated"] = W16.risk > ALPHA
wmean = lambda v, n: (v * n).sum() / n.sum()                   # share of group-windows over budget, weighted by group count
summ16 = W16.groupby("scheme", sort=False).apply(lambda g: pd.Series(dict(
    coverage=g.coverage.mean(), risk=g.risk.mean(), max_risk=g.risk.max(), windows_violated=g.violated.sum(),
    agent_windows_over=wmean(g.agent_viol, g.n_agent), history_windows_over=wmean(g.hist_viol, g.n_hist),
    repo_windows_over=wmean(g.repo_viol, g.n_repo), new_repo_coverage=g.new_repo_cov.mean(), new_repo_fail=g.new_repo_fail.mean())))
print("Group-conditional (hierarchical) thresholds, bi-weekly windows, alpha = 0.10:\n", summ16.round(3).to_string())
W16.to_csv("results_hierarchical_windows.csv", index=False); summ16.to_csv("results_hierarchical.csv")

## Step 17: Online risk control using early signals
In step 10 the online method adjusted its target using the realised risk of the window **two** windows back, because final outcomes take up to 14 days to arrive. By the time it reacted, the problem had passed.

But a team doesn't have to wait for the final outcome. Within a few days of an agent PR opening they can already see:
- whether it has been merged or closed;
- how many reviews it got, and whether any **requested changes** or **approved** it;
- how many comments humans left.

This step uses those early signals to **estimate** the risk of the window that just ended, and updates the target one window earlier:

1. For each window, train a small logistic-regression model on PRs whose final outcome was already known. It predicts the final outcome from the signals seen *D* days after the PR opened (*D* = 1, 3, 7), plus the repository prior.
2. Take the previous window's delegated issues that are at least *D* days old, as decided by that window's own model and threshold. Use the true outcome for those already merged or closed, and the predicted chance of success for the rest. This gives an estimate of that window's risk.
3. Update the target with the same rule as before: new target = old target + 0.5 × (0.10 − estimated risk), kept between 0.01 and 0.20.

For comparison, the output also shows static CRC, CRC with the 0.08 margin, the step 10 online method (two-window lag), and an **oracle** that knows the true risk one window back. The oracle is impossible in practice and serves only as a reference. A second table shows how close the early estimates are to the true risk (bias, mean absolute error, correlation).

**Output:** `results_online_early.csv`, `results_online_early_windows.csv`, `fig_online_early.png`.

In [ ]:
# Uses BW_STORE from step 10. Loads only the columns needed from the review and comment tables.
import pyarrow.parquet as pq, matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
ids = set(main.pr_id)
rev = pq.read_table(dl("pr_reviews.parquet"), columns=["pr_id", "state", "submitted_at", "user_type"]).to_pandas()
com = pq.read_table(dl("pr_comments.parquet"), columns=["pr_id", "created_at", "user_type"]).to_pandas()
rev, com = rev[rev.pr_id.isin(ids)], com[com.pr_id.isin(ids)]
print(f"PRs in the main set with any review: {main.pr_id.isin(rev.pr_id).mean():.3f}, with any comment: {main.pr_id.isin(com.pr_id).mean():.3f}")

pos = pd.Series(np.arange(len(main)), index=main.pr_id.values)
def event_times(df, col):
    t = pd.to_datetime(df[col], utc=True, errors="coerce")
    return pos.reindex(df.pr_id.values).values, ts(t)

def early_features(D):
    # What a team could see D days after each agent PR was opened.
    cut = DEC + D * DAY
    merged_by = (Y == 1) & (RES <= cut)
    closed_by = (Y == 0) & (RES <= cut)                        # closed without merge within D days
    f = {"merged_by": merged_by, "closed_by": closed_by}
    i, t = event_times(rev, "submitted_at"); ok = t <= cut[i]
    st_ = rev.state.str.upper().values
    for name, mask in [("reviews", np.ones(len(i), bool)), ("changes_requested", st_ == "CHANGES_REQUESTED"),
                       ("approved", st_ == "APPROVED")]:
        f[name] = np.bincount(i[ok & mask], minlength=len(main))
    i, t = event_times(com, "created_at"); human = com.user_type.fillna("").str.lower().values != "bot"
    f["human_comments"] = np.bincount(i[(t <= cut[i]) & human], minlength=len(main))
    F = np.column_stack([f[k] for k in f]).astype(float)
    F[:, 2:] = np.log1p(F[:, 2:])
    return F

def online_run(D, est):
    """est = 'lag2' (true risk, two windows back), 'early' (estimate from D-day signals, one window back),
    or 'oracle' (true risk one window back: not available in practice, shown as a reference)."""
    F = early_features(D) if est == "early" else None
    a, hist, out = ALPHA, [], []
    for w, st in enumerate(BW_STORE):
        p, ca, te = st["pm"], st["ca"], st["te"]
        t0 = DEC[te].min()
        if w >= 1 and est in ("early", "oracle") or w >= 2 and est == "lag2":
            prev = hist[-1] if est != "lag2" else hist[-2]
            if est == "oracle": r_hat = prev["risk"]
            elif est == "lag2": r_hat = prev["risk"]
            else:
                # early-outcome model: trained on PRs whose final outcome was known before this window,
                # using the D-day signals plus the repository prior (no fitted scores, so nothing leaks)
                trn = np.where(RES < t0)[0]
                em = LogisticRegression(max_iter=1000).fit(np.column_stack([F[trn], REPO_PRIOR[trn]]), Y[trn])
                # previous-window issues old enough to have D days of signals, scored by that window's own model
                old = DEC[prev["te"]] + D * DAY <= t0
                seen, dlg_prev = prev["te"][old], prev["delegated"][old]
                q = em.predict_proba(np.column_stack([F[seen], REPO_PRIOR[seen]]))[:, 1]
                q = np.where(RES[seen] < t0, Y[seen], q)                # outcomes already final by now are simply known
                r_hat = np.mean(dlg_prev * (1 - q)) if len(seen) else ALPHA
                prev["r_hat"] = r_hat
            a = float(np.clip(a + GAMMA * (ALPHA - r_hat), 0.01, 0.20))
        lam = lam_crc(p[ca], Y[ca], a)
        s = stats(p[te], Y[te], lam)
        hist.append(dict(te=te, delegated=p[te] >= lam, risk=s["risk"]))
        out.append(dict(window=st["window"], D=D, estimate=est, target=a, **s))
    for o, h in zip(out, hist): o["r_hat"] = h.get("r_hat", np.nan)
    return out

rows17 = []
for D in [1, 3, 7]: rows17 += online_run(D, "early")
rows17 += online_run(0, "lag2") + online_run(0, "oracle")
W17 = pd.DataFrame(rows17); W17["violated"] = W17.risk > ALPHA
W17["policy"] = W17.apply(lambda r: f"Online, early signals after {r.D} d" if r.estimate == "early"
                          else ("Online, true risk 2 windows back (step 10)" if r.estimate == "lag2"
                                else "Online, true risk 1 window back (oracle)"), axis=1)
base = pd.DataFrame([dict(window=st["window"], policy=name, **stats(st["pm"][st["te"]], Y[st["te"]], lam_crc(st["pm"][st["ca"]], Y[st["ca"]], a)))
                     for st in BW_STORE for name, a in [("Static CRC", ALPHA), ("Static CRC, margin 0.08", 0.08)]])
base["violated"] = base.risk > ALPHA
ALL17 = pd.concat([base, W17], ignore_index=True)
summ17 = ALL17.groupby("policy", sort=False).agg(coverage=("coverage", "mean"), risk=("risk", "mean"), max_risk=("risk", "max"),
                                                 windows_violated=("violated", "sum"), fail_among_delegated=("cond_fail", "mean"))
print("Online risk control with early signals, bi-weekly windows, alpha = 0.10:\n", summ17.round(3).to_string())

# How good is the early estimate of last window's risk?
acc = []
for D in [1, 3, 7]:
    e = W17[(W17.estimate == "early") & (W17.D == D)].reset_index(drop=True)
    est, true = e.r_hat.values, e.risk.values                          # estimate made one window later, vs the truth
    ok = ~np.isnan(est)
    acc.append(dict(D=D, windows=ok.sum(), bias=np.mean(est[ok] - true[ok]), mae=np.mean(np.abs(est[ok] - true[ok])),
                    corr=np.corrcoef(est[ok], true[ok])[0, 1]))
print("\nAccuracy of the early estimate of the previous window's risk:\n", pd.DataFrame(acc).round(3).to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 3.2))
for name, c, ls in [("Static CRC", "#2ca02c", "-"), ("Online, true risk 2 windows back (step 10)", "#1f77b4", "--"),
                    ("Online, early signals after 3 d", "#9467bd", "-"), ("Static CRC, margin 0.08", "#7f7f7f", ":")]:
    d_ = ALL17[ALL17.policy == name]; ax.plot(d_.window, d_.risk, marker="o", ms=3, color=c, ls=ls, label=name)
ax.axhline(ALPHA, color="k", ls="--", lw=.8); ax.set_ylabel("Realised risk"); ax.tick_params(axis="x", rotation=60, labelsize=7)
ax.grid(alpha=.3); ax.legend(fontsize=6.5, loc="upper center", bbox_to_anchor=(.5, -.42), ncol=2, frameon=False)
plt.savefig("fig_online_early.png", dpi=300, bbox_inches="tight"); plt.show()
ALL17.to_csv("results_online_early_windows.csv", index=False); summ17.to_csv("results_online_early.csv")


## Step 18: Asymmetric online risk control
Step 17 showed that early review signals reveal the previous window's risk almost exactly within a day, yet online control still failed. The rule raised its target in every calm window, so when risk jumped in mid-November it was already delegating more than usual and had no buffer.

This step keeps the same early-signal estimate (signals one day old) but changes **how the target reacts**:

| Rule | When risk was above budget | When risk was below budget | Highest target allowed |
|---|---|---|---|
| Symmetric (step 17) | step size 0.5 | step size 0.5 | 0.20 |
| Asymmetric | step size 1.0 (tighten fast) | step size 0.1 (loosen slowly) | 0.20 |
| Asymmetric, never above budget | 1.0 | 0.1 | 0.10 |
| Margin + online | 1.0 | 0.1 | 0.08 (starts at 0.08) |
| Margin + online, fast recovery | 1.0 | 0.5 | 0.08 (starts at 0.08) |

The last two combine the step 10 safety margin with online correction. They start at 0.08, tighten when the estimated risk goes over 0.10, and drift back towards 0.08 but never above it. In other words, the rule may only become *more* careful than the margin, never less.

The output shows coverage, risk, worst window and violated windows for each rule next to static CRC and the static margin, plus each rule's target in every window. The plot is saved as `fig_online_asymmetric.png`.

**Output:** `results_online_asymmetric.csv`, `results_online_asymmetric_windows.csv`, `fig_online_asymmetric.png`.

In [ ]:
# Uses BW_STORE (step 10) and early_features / rev / com from step 17, so run step 17 first.
D_EARLY = 1                                                     # signals one day old: the most accurate in step 17

def online_rule(gain_tighten, gain_loosen, start, cap, D=D_EARLY):
    # Same update as step 17, but tightening and loosening can use different step sizes,
    # and the target is never allowed above `cap`.
    F = early_features(D)
    a, prev, out = start, None, []
    for st in BW_STORE:
        p, ca, te = st["pm"], st["ca"], st["te"]
        t0 = DEC[te].min()
        if prev is not None:
            trn = np.where(RES < t0)[0]
            em = LogisticRegression(max_iter=1000).fit(np.column_stack([F[trn], REPO_PRIOR[trn]]), Y[trn])
            old = DEC[prev["te"]] + D * DAY <= t0
            seen, dlg_prev = prev["te"][old], prev["delegated"][old]
            q = em.predict_proba(np.column_stack([F[seen], REPO_PRIOR[seen]]))[:, 1]
            q = np.where(RES[seen] < t0, Y[seen], q)
            r_hat = np.mean(dlg_prev * (1 - q)) if len(seen) else ALPHA
            gain = gain_tighten if r_hat > ALPHA else gain_loosen
            a = float(np.clip(a + gain * (ALPHA - r_hat), 0.01, cap))
        lam = lam_crc(p[ca], Y[ca], a)
        out.append(dict(window=st["window"], target=a, **stats(p[te], Y[te], lam)))
        prev = dict(te=te, delegated=p[te] >= lam)
    return out

RULES = {"Symmetric (step 17): 0.5 / 0.5, cap 0.20":            (0.5, 0.5, ALPHA, 0.20),
         "Asymmetric: tighten 1.0, loosen 0.1, cap 0.20":       (1.0, 0.1, ALPHA, 0.20),
         "Asymmetric, never above budget: cap 0.10":            (1.0, 0.1, ALPHA, 0.10),
         "Margin + online: start 0.08, never above 0.08":       (1.0, 0.1, 0.08, 0.08),
         "Margin + online, fast recovery: loosen 0.5, cap 0.08": (1.0, 0.5, 0.08, 0.08)}
rows18 = []
for name, (gt, gl, a0, cap) in RULES.items():
    rows18 += [dict(policy=name, **r) for r in online_rule(gt, gl, a0, cap)]
W18 = pd.concat([base[base.policy.isin(["Static CRC", "Static CRC, margin 0.08"])], pd.DataFrame(rows18)], ignore_index=True)
W18["violated"] = W18.risk > ALPHA
summ18 = W18.groupby("policy", sort=False).agg(coverage=("coverage", "mean"), risk=("risk", "mean"), max_risk=("risk", "max"),
                                               windows_violated=("violated", "sum"), fail_among_delegated=("cond_fail", "mean"))
print(f"Asymmetric online control (early signals after {D_EARLY} day), bi-weekly windows, alpha = 0.10:\n", summ18.round(3).to_string())
print("\nTargets by window:\n", W18.dropna(subset=["target"]).pivot(index="window", columns="policy", values="target").round(3).to_string())

fig, ax = plt.subplots(figsize=(7, 3.2))
for name, c, ls in [("Static CRC", "#2ca02c", "-"), ("Static CRC, margin 0.08", "#7f7f7f", ":"),
                    ("Symmetric (step 17): 0.5 / 0.5, cap 0.20", "#1f77b4", "--"),
                    ("Asymmetric, never above budget: cap 0.10", "#ff7f0e", "-"),
                    ("Margin + online: start 0.08, never above 0.08", "#9467bd", "-")]:
    d_ = W18[W18.policy == name]; ax.plot(d_.window, d_.risk, marker="o", ms=3, color=c, ls=ls, label=name)
ax.axhline(ALPHA, color="k", ls="--", lw=.8); ax.set_ylabel("Realised risk"); ax.tick_params(axis="x", rotation=60, labelsize=7)
ax.grid(alpha=.3); ax.legend(fontsize=6.5, loc="upper center", bbox_to_anchor=(.5, -.42), ncol=2, frameon=False)
plt.savefig("fig_online_asymmetric.png", dpi=300, bbox_inches="tight"); plt.show()
W18.to_csv("results_online_asymmetric_windows.csv", index=False); summ18.to_csv("results_online_asymmetric.csv")

## Step 19: Repository-level information and project-level decisions
Two ideas, tested together.

**1. More history for "new" repositories.** Our history features count only agent PRs linked to an issue, about 30,000 of the 361,000 agent PRs in AIDev. A repository with no issue-linked history may still have many other agent PRs whose outcome is already known. This step computes a **broader repository history** from *all* agent PRs in the repository: the smoothed 14-day merge rate and the number of known outcomes, again using only outcomes known before the decision. It prints how many "new" issues actually have such history.

**2. Information about the repository itself.** From AIDev's `repository` table it adds log stars, log forks, whether the repository has a licence, and its main language (top 8). These are snapshots taken when AIDev was collected, not at decision time, which is a small limitation. Test coverage and contribution rules aren't in AIDev; they would need cloning each repository.

Three models are then compared in the bi-weekly windows, all with CRC at α = 0.10:

| Model | Features |
|---|---|
| Main model (as before) | repository history, issue structure, text score |
| Main model + repository information | the above + broader history + repository metadata |
| Project-level model | repository features only: no issue features at all, so every issue in a repository gets the same score at a given time |

The project-level model tests the idea of deciding **how much to delegate in a repository** rather than which issue. If it does as well as the main model, the issue features add nothing to the decision.

The output reports coverage, risk and violations, pooled AUC, AUC on new repositories and within-repository AUC. A second table gives results by repository history, which is where repository information should matter most.

**Output:** `results_repo_level.csv`, `results_repo_level_by_history.csv`, `results_repo_level_windows.csv`.

In [ ]:
# Uses BW_STORE (step 10). Reloads the PR table and the repository table.
hist_bin = lambda h: pd.cut(h, [-1, 0, 10, 50, np.inf], labels=["new (0)", "1-10", "11-50", ">50"])
def within_repo_auc(p, y, repos, min_n=20):
    aucs = [roc_auc_score(y[ix], p[ix]) for ix in pd.Series(np.arange(len(y))).groupby(repos).indices.values()
            if len(ix) >= min_n and 0 < y[ix].mean() < 1]
    return np.mean(aucs), len(aucs)
# 1) Broader agent history: every agent PR in the repository, not only the ones linked to an issue.
allpr = pd.read_parquet(dl("pull_request.parquet"), columns=["id", "repo_id", "created_at", "merged_at", "closed_at"]).drop_duplicates("id")
for c in ["created_at", "merged_at", "closed_at"]: allpr[c] = pd.to_datetime(allpr[c], utc=True, errors="coerce")
dl14 = allpr.created_at + pd.Timedelta(days=H)
ok14 = allpr.merged_at.notna() & (allpr.merged_at <= dl14)
allpr["y14"] = ok14.astype(float)
allpr["known"] = allpr.merged_at.where(ok14, allpr.closed_at.where(allpr.closed_at < dl14, dl14))
allpr = allpr[allpr.known.notna()]
print(f"All agent PRs: {len(allpr):,} in {allpr.repo_id.nunique():,} repositories "
      f"(the main set uses {len(main):,} issue-linked PRs in {main.repo_id.nunique():,} repositories)")

def broad_before(dec):
    # For each main-set issue: agent PRs in the same repository whose 14-day outcome was known before `dec`.
    s, c = np.zeros(len(main)), np.zeros(len(main))
    kn, yy = ts(allpr.known), allpr.y14.values
    by_repo = pd.Series(np.arange(len(allpr))).groupby(allpr.repo_id.values).indices
    for rp, idx in pd.Series(np.arange(len(main))).groupby(main.repo_id.values).indices.items():
        j = by_repo.get(rp)
        if j is None: continue
        o = np.argsort(kn[j]); k_ = kn[j][o]; cy = np.concatenate([[0.0], np.cumsum(yy[j][o])])
        hi = np.searchsorted(k_, dec[idx], side="left")
        s[idx], c[idx] = cy[hi], hi
    return s, c

sb, cb = broad_before(DEC)
BROAD_PRIOR, BROAD_N = (sb + 10 * G) / (cb + 10), np.log1p(cb)
newm = REPO_HIST == 0
print(f"Issues from repositories with no issue-linked history: {newm.mean():.3f}; "
      f"of these, {np.mean(cb[newm] > 0):.3f} already have other agent PRs with known outcomes (median {np.median(cb[newm]):.0f})")

# 2) Repository metadata (a snapshot taken when AIDev was collected, not at decision time)
try:
    repo = pd.read_parquet(dl("repository.parquet"))
    repo = repo.rename(columns={"id": "repo_id"}).drop_duplicates("repo_id").set_index("repo_id")
    m_ = repo.reindex(main.repo_id.values)
    top = m_.language.value_counts().index[:8]
    META = np.column_stack([np.log1p(m_.stars.fillna(0)), np.log1p(m_.forks.fillna(0)), m_.license.notna()] +
                           [(m_.language == l).values for l in top]).astype(float)
    META_NAMES = ["log stars", "log forks", "has licence"] + [f"lang: {l}" for l in top]
    print(f"Repository metadata found for {m_.stars.notna().mean():.3f} of issues")
except Exception as e:
    print("Repository table not available, continuing without metadata:", e)
    META, META_NAMES = np.zeros((len(main), 0)), []

REPO_INFO = np.column_stack([BROAD_PRIOR, BROAD_N, META])
def auc_or_nan(y, p): return roc_auc_score(y, p) if 0 < y.mean() < 1 else np.nan

# 3) Bi-weekly evaluation
rows19, recs19 = [], []
for st in BW_STORE:
    tr, ca, te, TXT = st["tr"], st["ca"], st["te"], st["TXT"]
    Xm = np.column_stack([REPO_PRIOR, REPO_N, REPO_PRIOR90, ISS, TXT])
    pols = {"Main model (as before)": st["pm"]}
    Xr = np.column_stack([Xm, REPO_INFO]);                         pols["Main model + repository information"] = lgbm(Xr, tr).predict_proba(Xr)[:, 1]
    Xp = np.column_stack([REPO_PRIOR, REPO_N, REPO_PRIOR90, REPO_INFO]); pols["Project-level model (no issue features)"] = lgbm(Xp, tr).predict_proba(Xp)[:, 1]
    for name, p in pols.items():
        lam = lam_crc(p[ca], Y[ca], ALPHA); dlg = p[te] >= lam; nw = newm[te]
        rows19.append(dict(window=st["window"], policy=name, auc=roc_auc_score(Y[te], p[te]), auc_new=auc_or_nan(Y[te][nw], p[te][nw]),
                           **stats(p[te], Y[te], lam), new_cov=dlg[nw].mean(),
                           new_fail=1 - Y[te][dlg & nw].mean() if (dlg & nw).any() else np.nan))
        recs19.append(pd.DataFrame(dict(policy=name, y=Y[te], p=p[te], delegated=dlg, repo=main.repo_id.values[te], history=hist_bin(REPO_HIST[te]))))

W19 = pd.DataFrame(rows19); W19["violated"] = W19.risk > ALPHA
R19 = pd.concat(recs19, ignore_index=True)
summ19 = W19.groupby("policy", sort=False).agg(coverage=("coverage", "mean"), risk=("risk", "mean"), max_risk=("risk", "max"),
                                               windows_violated=("violated", "sum"), fail_among_delegated=("cond_fail", "mean"))
pooled19 = R19.groupby("policy", sort=False).apply(lambda g: pd.Series(dict(
    auc=roc_auc_score(g.y, g.p), auc_new_repos=auc_or_nan(g.y[g.history == "new (0)"].values, g.p[g.history == "new (0)"].values),
    within_repo_auc=within_repo_auc(g.p.values, g.y.values, g.repo.values)[0])))
print("\nRepository-level information, bi-weekly windows, alpha = 0.10:\n", summ19.join(pooled19).round(3).to_string())
by_hist19 = R19.groupby(["policy", "history"], observed=True, sort=False).apply(lambda g: pd.Series(dict(
    coverage=g.delegated.mean(), fail_among_delegated=(1 - g.y[g.delegated].mean()) if g.delegated.any() else np.nan,
    auc=auc_or_nan(g.y.values, g.p.values))))
print("\nBy repository history (issue-linked outcomes known):\n", by_hist19.unstack("history").round(3).to_string())
summ19.join(pooled19).to_csv("results_repo_level.csv"); by_hist19.to_csv("results_repo_level_by_history.csv")
W19.to_csv("results_repo_level_windows.csv", index=False)

## Step 20: Ablation of repository information and the combined policy
Step 19 added two kinds of repository information at once. The **broader agent history** only uses outcomes known before each decision. The **repository metadata** (stars, forks, licence, language) is a snapshot taken when AIDev was collected, which may be later than the decision. This step separates them, to show how much of the gain is time-safe.

**Part 1, ablation:** the main model alone, plus broader history only, plus metadata only, and plus both, all with static CRC at 0.10.

**Part 2, combined policy:** the main model, main + broader history and main + both, each with the two best safeguards: the 0.08 margin, and margin + online (step 18's best rule, which may tighten but never loosen past 0.08).

Every row reports coverage, risk, worst window, violated windows, coverage and failure rate in new repositories, pooled AUC, AUC on new repositories and within-repository AUC.

**Output:** `results_ablation_combined.csv`.

In [ ]:
# Step 20: which part of the repository information helps, and the combined policy.
# Needs steps 0, 1, 2, 10, 17 and 19 in this session (BW_STORE, early_features, BROAD_PRIOR, META, ...).
BROAD = np.column_stack([BROAD_PRIOR, BROAD_N])
VARIANTS = {"Main model":                         None,
            "Main + broader history only":        BROAD,
            "Main + metadata only (stars, ...)":  META,
            "Main + both":                        np.column_stack([BROAD, META])}

# Score every window once with each variant
SC = {name: [] for name in VARIANTS}
for st in BW_STORE:
    Xm = np.column_stack([REPO_PRIOR, REPO_N, REPO_PRIOR90, ISS, st["TXT"]])
    for name, extra in VARIANTS.items():
        if extra is None: SC[name].append(st["pm"]); continue
        X_ = np.column_stack([Xm, extra]); SC[name].append(lgbm(X_, st["tr"]).predict_proba(X_)[:, 1])

def run_policy(scores, mode, D=1):
    # mode: "static" (CRC at 0.10), "margin" (CRC at 0.08) or "margin+online" (step 18's best rule)
    F = early_features(D) if mode == "margin+online" else None
    a, prev, out, rec = (0.08 if mode != "static" else ALPHA), None, [], []
    for st, p in zip(BW_STORE, scores):
        ca, te = st["ca"], st["te"]; t0 = DEC[te].min()
        if mode == "margin+online" and prev is not None:
            trn = np.where(RES < t0)[0]
            em = LogisticRegression(max_iter=1000).fit(np.column_stack([F[trn], REPO_PRIOR[trn]]), Y[trn])
            old = DEC[prev["te"]] + D * DAY <= t0
            seen, dlg_prev = prev["te"][old], prev["delegated"][old]
            q = np.where(RES[seen] < t0, Y[seen], em.predict_proba(np.column_stack([F[seen], REPO_PRIOR[seen]]))[:, 1])
            r_hat = np.mean(dlg_prev * (1 - q)) if len(seen) else ALPHA
            a = float(np.clip(a + (1.0 if r_hat > ALPHA else 0.1) * (ALPHA - r_hat), 0.01, 0.08))
        lam = lam_crc(p[ca], Y[ca], a); dlg = p[te] >= lam; nw = newm[te]
        out.append(dict(window=st["window"], **stats(p[te], Y[te], lam), new_cov=dlg[nw].mean(),
                        new_fail=1 - Y[te][dlg & nw].mean() if (dlg & nw).any() else np.nan))
        rec.append(pd.DataFrame(dict(y=Y[te], p=p[te], repo=main.repo_id.values[te], new=nw)))
        prev = dict(te=te, delegated=dlg)
    return pd.DataFrame(out), pd.concat(rec, ignore_index=True)

def summarise(name, mode, W, R):
    W = W.assign(violated=W.risk > ALPHA)
    return dict(model=name, threshold=mode, coverage=W.coverage.mean(), risk=W.risk.mean(), max_risk=W.risk.max(),
                windows_violated=int(W.violated.sum()), fail_among_delegated=W.cond_fail.mean(),
                new_repo_coverage=W.new_cov.mean(), new_repo_fail=W.new_fail.mean(),
                auc=roc_auc_score(R.y, R.p), auc_new_repos=auc_or_nan(R.y[R.new].values, R.p[R.new].values),
                within_repo_auc=within_repo_auc(R.p.values, R.y.values, R.repo.values)[0])

# Part 1: ablation, all with static CRC at 0.10
rows21 = [summarise(name, "static", *run_policy(SC[name], "static")) for name in VARIANTS]
# Part 2: combined policy, the main model and the best variant with the two best safeguards
for name in ["Main model", "Main + broader history only", "Main + both"]:
    for mode in ["margin", "margin+online"]:
        rows21.append(summarise(name, mode, *run_policy(SC[name], mode)))
T21 = pd.DataFrame(rows21).set_index(["model", "threshold"])
print("Ablation and combined policy, 14 bi-weekly windows, budget 0.10:\n", T21.round(3).to_string())
T21.to_csv("results_ablation_combined.csv")


## Step 21: Save everything
Zips all `results_*.csv` and `fig*.png` files into `revision_results.zip` and copies everything, plus the two parquet datasets, to `MyDrive/agent_routing`. The blank label sample is saved as `label_validation_sample_BLANK.csv`, so it can never overwrite a sheet you've already coded.

In [ ]:
# Collect every result file into one zip and copy it to Google Drive.
# The blank label-validation sample goes to its own name so it can never
# overwrite a sheet you have already coded.
import glob, os, zipfile, shutil
files = sorted(glob.glob("results_*.csv") + glob.glob("fig*.png"))
if os.path.exists("label_validation_sample.csv"):
    shutil.copy("label_validation_sample.csv", "label_validation_sample_BLANK.csv"); files.append("label_validation_sample_BLANK.csv")
with zipfile.ZipFile("revision_results.zip", "w") as z:
    for f in files: z.write(f)
print("Zipped", len(files), "files into revision_results.zip")
from google.colab import drive
drive.mount("/content/drive")
os.makedirs("/content/drive/MyDrive/agent_routing", exist_ok=True)
for f in files + ["revision_results.zip", "routing_main_v2.parquet", "routing_all_v2.parquet"]:
    if os.path.exists(f): shutil.copy(f, "/content/drive/MyDrive/agent_routing/")
print("Copied to MyDrive/agent_routing")